# NB03 — Preprocesamiento y Feature Engineering

**Proyecto:** Predicción de siniestralidad vial en El Salvador (`siniestralidad-vial-sv`)
**Etapa 1**

---

## 1. Objetivo del notebook

Este notebook transforma los datos **crudos validados en NB01** y **caracterizados en NB02**
en las matrices de features listas para modelar en NB04. No introduce datos nuevos: deriva
todas las variables a partir de las seis fuentes ya cargadas por `src/config.py`.

### 1.1 Decisiones heredadas de NB02 que aquí se materializan

El EDA dejó cerradas las siguientes decisiones de diseño, que este notebook implementa:

- **Unidad de conteo:** grilla `distrito × fecha × franja_horaria` con celdas en cero
  (la mayoría de combinaciones no registran siniestros; el cero es información, no ausencia).
- **Calendario:** `mes`, temporada lluviosa (May–Oct), fin de semana (vie/sáb),
  `franja_horaria`, feriado y período agostino. Los **feriados locales aplican solo a su distrito**.
- **Colinealidad:** no se incluyen simultáneamente `mes` y `es_lluviosa` (r ≈ 0.32 en NB02).
- **Exposición:** el denominador de riesgo combina población residente (Censo 2024) con
  parque vehicular; se señalan/filtran distritos de población muy baja.
- **Clima:** se usa a resolución **estacional/mensual**, no precipitación diaria puntual,
  reutilizando el join `distrito → estación` por cercanía definido en NB02
  (Ilopango para el AMSS por la baja completitud de la estación SS).
- **Categóricas:** se agrupa la cola larga de `factor_causante` y las categorías
  minoritarias de `tipo_siniestro` / `tipo_vehiculo`.

### 1.2 Dos matrices de salida (dos unidades de análisis)

A diferencia de un pipeline de tarea única, NB03 produce **dos matrices** porque los dos
targets del proyecto viven en unidades distintas:

| | **Matriz A — Frecuencia** | **Matriz B — Severidad** |
|---|---|---|
| **Unidad** | celda `distrito × fecha × franja` | siniestro individual |
| **Target** | `n_siniestros` (conteo) | `nivel_riesgo` (3 clases) |
| **Modelo (NB04)** | Regresión Poisson / Binomial Negativa | Clasificación |
| **Features** | solo **ex-ante** (calendario, exposición, clima, ubicación) | **ex-post** admitidas (condicional a que el siniestro ocurrió) |

**Nota metodológica — por qué dos matrices y no una.**
Variables como `factor_causante`, `tipo_siniestro`, `tipo_vehiculo` o `condicion_via`
solo se conocen **después** de que el siniestro ocurrió. Usarlas para predecir la
*frecuencia* ex-ante sería fuga de información (leakage). En cambio, para predecir la
*severidad* son predictores legítimos: el modelo de *injury severity* condiciona a que
el siniestro ya pasó y estima su gravedad. Por eso las variables ex-post se restringen
a la Matriz B y quedan **excluidas** de la Matriz A.

### 1.3 Productos que este notebook deja en `data/processed/`

- `matriz_frecuencia.parquet` — grilla completa con features ex-ante y `n_siniestros`.
- `matriz_severidad.parquet` — un registro por siniestro con features ex-post y `nivel_riesgo`.
- Cortes `train` / `test` por partición **temporal** (sin fuga de períodos futuros).

In [1]:
# _find_root: agrega la raíz del repo al sys.path para poder importar src/config.py
# (patrón fijo del proyecto — no redefinir rutas ni seed aquí; todo vive en config.py)
import sys
from pathlib import Path


def _find_root(marker: str = "src") -> Path:
    """Sube directorios desde el cwd hasta encontrar la raíz del repo (la que contiene src/)."""
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(
        f"No se encontró la raíz del repo (carpeta que contenga '{marker}/') "
        f"partiendo de {Path.cwd()}"
    )


ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config as cfg

print(f"Raíz del repo: {ROOT}")
print(f"Semilla global: {cfg.SEED}")
print(f"Ruta datos crudos:      {cfg.RAW_DIR}")
print(f"Ruta datos procesados:  {cfg.PROCESSED_DIR}")

Raíz del repo: /Users/jpurquilla/Desktop/Trabajo de Grado/Etapa 1/siniestralidad-vial-sv
Semilla global: 42
Ruta datos crudos:      /Users/jpurquilla/Desktop/Trabajo de Grado/Etapa 1/siniestralidad-vial-sv/data/raw
Ruta datos procesados:  /Users/jpurquilla/Desktop/Trabajo de Grado/Etapa 1/siniestralidad-vial-sv/data/processed


## 2. Carga de las fuentes crudas

Se cargan las seis fuentes del proyecto **exclusivamente a través de los loaders de
`src/config.py`**, para garantizar que NB03 usa exactamente los mismos datos, parseos y
limpiezas validados en NB01 y NB02. No se lee ningún archivo con rutas directas ni se
re-implementa ninguna transformación de carga.

| Fuente | Loader | Rol en NB03 |
|---|---|---|
| Siniestros | `cfg.load_siniestros()` | Base de agregación de la **Matriz A** y unidad de la **Matriz B** |
| Víctimas | `cfg.load_victimas()` | Severidad y `grupo_vulnerable` / `rango_etario` (Matriz B) |
| Censo 2024 | `cfg.load_censo()` | Población residente por distrito (exposición) |
| Estaciones | `cfg.load_estaciones()` | Catálogo para el join `distrito → estación` |
| Feriados | `cfg.load_feriados()` | Calendario: feriados nacionales y **locales por distrito** |
| Clima | `cfg.load_clima_all()` | Precipitación horaria → se agrega a resolución estacional/mensual |



In [2]:
# =============================================================
# Sección 2 — Carga de las fuentes crudas
# Carga de las 6 fuentes vía loaders de config.py (sin rutas directas)
# =============================================================
siniestros  = cfg.load_siniestros()
victimas    = cfg.load_victimas()
censo       = cfg.load_censo()
estaciones  = cfg.load_estaciones()
feriados    = cfg.load_feriados()
clima       = cfg.load_clima_all()

print("Fuentes cargadas — dimensiones (filas, columnas):")
print(f"  siniestros : {siniestros.shape}")
print(f"  victimas   : {victimas.shape}")
print(f"  censo      : {censo.shape}")
print(f"  estaciones : {estaciones.shape}")
print(f"  feriados   : {feriados.shape}")
print(f"  clima      : {clima.shape}")

Fuentes cargadas — dimensiones (filas, columnas):
  siniestros : (89946, 23)
  victimas   : (60402, 9)
  censo      : (262, 7)
  estaciones : (7, 8)
  feriados   : (70, 8)
  clima      : (225227, 24)


### 2.1 Qué se verifica al cargar

Antes de derivar variables se confirma que cada fuente llega con la forma esperada.
Este bloque es de **verificación de integridad**, no de transformación:

- **Siniestros:** columna `fecha` parseada como `datetime` y `franja_horaria` presente;
  rango temporal 2022-01-01 … 2026-06-30; 103 distritos.
- **Víctimas:** enlazables por `id_siniestro`, sin registros huérfanos.
- **Censo:** el **nombre de distrito por sí solo NO es clave única** — existen 11 nombres
  homónimos en departamentos distintos (p. ej. *El Carmen* en Cuscatlán y La Unión).
  La clave válida es la terna `(departamento, municipio, distrito)`, verificada como
  única (262 filas = 262 combinaciones). `poblacion` es numérica.
- **Clima:** columna temporal unificada `time` tras fusionar los dos formatos Meteostat.

> **Hallazgo de integridad.** La verificación del censo obligó a corregir el supuesto
> inicial de que el nombre de distrito era único. No lo es: la homonimia entre
> departamentos es real y esperada en la división territorial salvadoreña. Esto define
> la clave de join usada en la Sección 2.2 y siguientes.

In [6]:
# =============================================================
# Sección 2.1 — Verificación de integridad de carga
# (no transforma datos; solo confirma la forma esperada de cada fuente)
# =============================================================
import pandas as pd

print("=" * 60)
print("VERIFICACIÓN DE INTEGRIDAD DE CARGA")
print("=" * 60)

# --- Siniestros: fecha datetime, franja presente, rango temporal esperado ---
assert pd.api.types.is_datetime64_any_dtype(siniestros["fecha"]), \
    "siniestros['fecha'] no está parseada como datetime"
assert "franja_horaria" in siniestros.columns, "falta 'franja_horaria' en siniestros"

fmin, fmax = siniestros["fecha"].min(), siniestros["fecha"].max()
print(f"\n[Siniestros]")
print(f"  Rango temporal : {fmin.date()} … {fmax.date()}")
print(f"  Franjas        : {sorted(siniestros['franja_horaria'].unique())}")
print(f"  Distritos      : {siniestros['distrito'].nunique()}")

# --- Víctimas: enlazables por id_siniestro (sin huérfanas) ---
ids_sin = set(siniestros["id_siniestro"])
huerfanas = ~victimas["id_siniestro"].isin(ids_sin)
print(f"\n[Víctimas]")
print(f"  Registros            : {len(victimas):,}")
print(f"  Huérfanas (sin match): {int(huerfanas.sum())}")
assert huerfanas.sum() == 0, "hay víctimas con id_siniestro inexistente en siniestros"

# --- Censo: la CLAVE COMPUESTA (dep, mun, dist) es única; poblacion numérica ---
# Nota: el nombre de distrito por sí solo NO es único (homonimia entre departamentos,
# p. ej. 'El Carmen' en Cuscatlán y La Unión). La clave válida es la terna completa.
clave_censo = ["departamento", "municipio", "distrito"]
print(f"\n[Censo]")
print(f"  Filas                      : {len(censo)}")
print(f"  Distritos (nombre único)   : {censo['distrito'].nunique()}")
print(f"  Combinaciones (dep,mun,dist): {censo[clave_censo].drop_duplicates().shape[0]}")
print(f"  poblacion es numérica      : {pd.api.types.is_numeric_dtype(censo['poblacion'])}")
assert censo[clave_censo].duplicated().sum() == 0, \
    "la clave compuesta (departamento, municipio, distrito) tiene duplicados en el censo"

# --- Clima: columna temporal unificada 'time' ---
print(f"\n[Clima]")
print(f"  Columnas: {list(clima.columns)}")
assert "time" in clima.columns, "clima no tiene la columna temporal unificada 'time'"
print(f"  Rango temporal: {clima['time'].min()} … {clima['time'].max()}")

print("\n" + "=" * 60)
print("OK — todas las verificaciones de carga pasaron")
print("=" * 60)

VERIFICACIÓN DE INTEGRIDAD DE CARGA

[Siniestros]
  Rango temporal : 2022-01-01 … 2026-06-30
  Franjas        : ['Madrugada (00-05)', 'Manana (06-11)', 'Noche (18-23)', 'Tarde (12-17)']
  Distritos      : 103

[Víctimas]
  Registros            : 60,402
  Huérfanas (sin match): 0

[Censo]
  Filas                      : 262
  Distritos (nombre único)   : 256
  Combinaciones (dep,mun,dist): 262
  poblacion es numérica      : True

[Clima]
  Columnas: ['time', 'station_id', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco', 'estacion', 'temp_source', 'rhum_source', 'prcp_source', 'wdir_source', 'wspd_source', 'wpgt_source', 'pres_source', 'cldc', 'cldc_source', 'coco_source']
  Rango temporal: 2022-01-01 00:00:00 … 2026-07-13 04:00:00

OK — todas las verificaciones de carga pasaron


### 2.2 Normalización y reconciliación de claves geográficas

La verificación de carga reveló que el nombre de distrito **no es una clave de join
fiable tal cual**, por dos motivos detectados empíricamente:

1. **Homonimia entre departamentos.** Existen 11 nombres de distrito repetidos en
   departamentos distintos (p. ej. *El Carmen* en Cuscatlán y en La Unión; *El Rosario*
   en Cuscatlán, La Paz y Morazán). El nombre solo no identifica un distrito; la clave
   única es la combinación `(departamento, municipio, distrito)`, verificada como única
   en el censo (262 filas = 262 combinaciones).

2. **Diferencias de escritura entre fuentes.** La tabla `siniestros` registra los
   nombres **sin tildes** ("Ahuachapan", "Colon", "Usulutan"), mientras que el censo
   los conserva ("Ahuachapán", "Colón", "Usulután"). Un join literal por nombre perdería
   silenciosamente 23 de 103 distritos (incluidos distritos clave del AMSS como
   Antiguo Cuscatlán), dejándolos sin exposición → tasas inválidas en la Matriz A.

**Solución adoptada.** Se usa `cfg.reconciliar_distrito()`, que normaliza los nombres a
una forma canónica (minúsculas, sin tildes, sin espacios extremos) y luego aplica un
mapeo de correcciones de nomenclatura. Sobre nombres normalizados, la cobertura sube de
80/103 a **102/103** sin introducir colisiones (ningún par de distritos distintos
colapsa al mismo nombre normalizado).

**Corrección manual de nomenclatura (reforma territorial 2023).** El único distrito
restante, `Dolores` (Cabañas, Cabañas Este), corresponde a **`Villa Dolores`** en el
censo: mismo departamento y municipio, nombre abreviado en la fuente de siniestros.
El mapeo `{"dolores": "villa dolores"}` (en `cfg.MAPEO_DISTRITOS`) lo resuelve. Con esta
corrección la cobertura es **103/103**.

**Clave de join definitiva para NB03:** distrito **reconciliado** + `departamento`
normalizado como desambiguador de homónimos. La lógica vive en `config.py`
(`cfg.norm_distrito`, `cfg.reconciliar_distrito`, `cfg.MAPEO_DISTRITOS`) para poder
reutilizarse en NB04. Esta clave se usa en las Secciones 6 (exposición) y 7 (clima).

In [7]:
# =============================================================
# Sección 2.2 — Normalización y reconciliación de claves geográficas
# Construye la clave de join distrito<->censo usando cfg.reconciliar_distrito
# (normaliza tildes + corrige nomenclatura: 'dolores' -> 'villa dolores').
# =============================================================

# Clave de join en cada tabla: distrito reconciliado + departamento normalizado.
# El departamento desambigua los 11 nombres de distrito homónimos entre deptos.
siniestros["distrito_key"]     = siniestros["distrito"].map(cfg.reconciliar_distrito)
siniestros["departamento_key"] = siniestros["departamento"].map(cfg.norm_distrito)

censo["distrito_key"]     = censo["distrito"].map(cfg.reconciliar_distrito)
censo["departamento_key"] = censo["departamento"].map(cfg.norm_distrito)

# --- Re-confirmar cobertura: ¿todo distrito de siniestros existe en el censo? ---
key_sin   = set(zip(siniestros["departamento_key"], siniestros["distrito_key"]))
key_censo = set(zip(censo["departamento_key"], censo["distrito_key"]))

sin_match = sorted(key_sin - key_censo)
print("RECONCILIACIÓN DE CLAVES GEOGRÁFICAS")
print("-" * 50)
print(f"  Distritos únicos en siniestros : {len(key_sin)}")
print(f"  Sin match en censo             : {len(sin_match)}")
if sin_match:
    print(f"  >> PENDIENTES: {sin_match}")
else:
    print("  >> Cobertura 103/103 — todos los distritos matchean.")

# Guardarrail: la construcción de exposición (Sección 6) depende de esto.
assert not sin_match, f"Distritos sin exposición posible: {sin_match}"

RECONCILIACIÓN DE CLAVES GEOGRÁFICAS
--------------------------------------------------
  Distritos únicos en siniestros : 103
  Sin match en censo             : 0
  >> Cobertura 103/103 — todos los distritos matchean.


## 3. Grilla espacio-temporal base (Matriz A)

La Matriz A modela la **frecuencia** de siniestros. Su unidad de análisis es la celda
`distrito × fecha × franja_horaria`. Para modelar conteos correctamente, la grilla debe
incluir **todas** las combinaciones posibles del período, no solo aquellas donde hubo
siniestros: las celdas con cero siniestros son información válida (ese distrito, ese día,
esa franja, no registró accidentes) y son mayoría. Omitirlas produciría un sesgo de
selección que inflaría artificialmente la tasa estimada.

### 3.1 Dimensiones de la grilla

| Dimensión | Valores | Cantidad |
|---|---|---|
| Distritos | distritos observados en `siniestros` | 103 |
| Fechas | días de 2022-01-01 a 2026-06-30 | 1 642 |
| Franjas | Madrugada / Mañana / Tarde / Noche | 4 |
| **Total celdas** | producto cartesiano | **676 504** |

### 3.2 Por qué 103 distritos y no todos los del país

La grilla se construye sobre los **103 distritos que aparecen en `siniestros`**, no sobre
los 262 distritos del Censo 2024. La razón es metodológica y debe quedar documentada:

- De las **262 combinaciones distrito-departamento** del Censo 2024, solo **103** aparecen
  en la fuente de siniestros; las **159** restantes **no tienen registros** en la fuente.
  Su ausencia es un **cero estructural por cobertura de la fuente**, no un cero observado.
  No sabemos si en esos distritos no hubo accidentes o si simplemente no fueron
  reportados/capturados en el dataset.
- Incluirlos como celdas en cero equivaldría a **afirmar como dato** que allí nunca
  ocurrió un siniestro en 1 642 días — una afirmación que la fuente no respalda. Eso
  introduciría ~1.04 millones de ceros no observados (159 × 1642 × 4), que dominarían
  el entrenamiento y sesgarían el modelo hacia predecir cero.
- Restringir la grilla a los 103 distritos observados mantiene la correspondencia
  **1:1 entre la unidad modelada y la evidencia disponible**: cada cero de la grilla es
  un cero real dentro de un distrito que sí está bajo observación.

**Evidencia que se deja en el notebook.** La celda de código siguiente documenta
explícitamente: (a) los 103 distritos incluidos, (b) el conteo de distritos del censo
que quedan fuera, y (c) la confirmación de que esos distritos fuera de la grilla tienen
efectivamente cero registros en `siniestros` (verificación de que la exclusión es por
ausencia en la fuente, no por un error de filtrado).

**Decisión abierta (Etapa 2).** Ampliar la cobertura más allá de los 103 distritos queda
como línea futura, condicionada a obtener una fuente de siniestros con cobertura nacional
completa. Mientras tanto, cualquier predicción del modelo se declara válida **solo para
los 103 distritos bajo observación**.

In [8]:
# =============================================================
# Sección 3 — Grilla espacio-temporal base (Matriz A)
# Producto cartesiano distrito × fecha × franja con TODAS las combinaciones.
# Las fechas y distritos se derivan del dato (no se hardcodean).
# =============================================================

# --- Ejes de la grilla, derivados de siniestros ---
distritos_grilla = sorted(siniestros["distrito_key"].unique())
fecha_min = siniestros["fecha"].min().normalize()
fecha_max = siniestros["fecha"].max().normalize()
fechas_grilla = pd.date_range(fecha_min, fecha_max, freq="D")
franjas_grilla = sorted(siniestros["franja_horaria"].unique())

n_dist, n_fecha, n_franja = len(distritos_grilla), len(fechas_grilla), len(franjas_grilla)

print("EJES DE LA GRILLA")
print("-" * 50)
print(f"  Distritos : {n_dist}")
print(f"  Fechas    : {n_fecha}  ({fecha_min.date()} … {fecha_max.date()})")
print(f"  Franjas   : {n_franja}  {franjas_grilla}")
print(f"  Celdas esperadas (producto): {n_dist * n_fecha * n_franja:,}")

# --- Producto cartesiano ---
grilla = pd.MultiIndex.from_product(
    [distritos_grilla, fechas_grilla, franjas_grilla],
    names=["distrito_key", "fecha", "franja_horaria"],
).to_frame(index=False)

print(f"\n  Grilla construida: {grilla.shape[0]:,} filas × {grilla.shape[1]} columnas")
assert grilla.shape[0] == n_dist * n_fecha * n_franja, "dimensión de grilla inesperada"
assert grilla.duplicated().sum() == 0, "la grilla tiene celdas duplicadas"

EJES DE LA GRILLA
--------------------------------------------------
  Distritos : 103
  Fechas    : 1642  (2022-01-01 … 2026-06-30)
  Franjas   : 4  ['Madrugada (00-05)', 'Manana (06-11)', 'Noche (18-23)', 'Tarde (12-17)']
  Celdas esperadas (producto): 676,504

  Grilla construida: 676,504 filas × 3 columnas


In [10]:
# =============================================================
# Sección 3.2 — Evidencia: por qué 103 distritos y no todos los del país
# Deja constancia ejecutada de que los distritos fuera de la grilla
# están ausentes en la FUENTE (cero estructural), no perdidos por un bug.
# Se cuenta con la CLAVE COMPUESTA (departamento_key, distrito_key),
# coherente con la clave de join del resto del notebook.
# =============================================================

# Claves compuestas en cada universo
censo_keys  = set(zip(censo["departamento_key"], censo["distrito_key"]))
grilla_keys = set(zip(siniestros["departamento_key"], siniestros["distrito_key"]))

# (a) Distritos incluidos en la grilla
print(f"(a) Distritos en la grilla (observados en siniestros): {len(grilla_keys)}")

# (b) Distritos del censo que quedan FUERA de la grilla (clave compuesta)
fuera = sorted(censo_keys - grilla_keys)
print(f"(b) Distritos del censo (clave compuesta dep+dist): {len(censo_keys)}")
print(f"    Distritos fuera de la grilla                  : {len(fuera)}")
print(f"    Verificación aritmética: {len(censo_keys)} - {len(grilla_keys)} = "
      f"{len(censo_keys) - len(grilla_keys)}")

# (c) Verificación: esos distritos fuera tienen CERO registros en siniestros
sin_keys = set(zip(siniestros["departamento_key"], siniestros["distrito_key"]))
registros_fuera = siniestros[
    [k in fuera for k in zip(siniestros["departamento_key"], siniestros["distrito_key"])]
].shape[0]
print(f"(c) Registros de siniestros en distritos fuera de la grilla: {registros_fuera}")
assert registros_fuera == 0, \
    "hay siniestros en distritos excluidos → la grilla perdió datos (revisar filtro)"

print("\n>> La exclusión es por AUSENCIA en la fuente (cero estructural), "
      "no por pérdida de datos.")

(a) Distritos en la grilla (observados en siniestros): 103
(b) Distritos del censo (clave compuesta dep+dist): 262
    Distritos fuera de la grilla                  : 159
    Verificación aritmética: 262 - 103 = 159
(c) Registros de siniestros en distritos fuera de la grilla: 0

>> La exclusión es por AUSENCIA en la fuente (cero estructural), no por pérdida de datos.


## 4. Agregación del target de conteo

Con la grilla base construida, se cuenta cuántos siniestros ocurren en cada celda
`(distrito, fecha, franja)` y se une el resultado a la grilla. Las celdas sin siniestros
—la gran mayoría— quedan en **0**, que es el valor correcto: representan ausencia
observada de accidentes en un distrito bajo observación.

### 4.1 Target principal: `n_siniestros`

`n_siniestros` es el conteo de accidentes por celda. Es el **target de regresión** de la
Matriz A, que en NB04 se modela con Regresión de Poisson y Binomial Negativa (la
sobredispersión detectada en NB02 —índice ≈ 2.7— justifica comparar ambas por AIC).



In [13]:
# =============================================================
# Sección 4 — Agregación del target de conteo n_siniestros
# Cuenta siniestros por celda (distrito, fecha, franja), une a la grilla
# (celdas sin siniestro -> 0), verifica integridad y diagnostica zero-inflation.
# =============================================================

# --- 4.1 Agregación por celda: conteo + severidad (ex-post, insumo descriptivo) ---
siniestros["_fecha_dia"] = siniestros["fecha"].dt.normalize()

agg = (
    siniestros
    .groupby(["distrito_key", "_fecha_dia", "franja_horaria"], observed=True)
    .agg(
        n_siniestros=("id_siniestro", "size"),
        fallecidos=("fallecidos", "sum"),
        lesionados=("lesionados", "sum"),
    )
    .reset_index()
    .rename(columns={"_fecha_dia": "fecha"})
)
print(f"Celdas con al menos 1 siniestro: {len(agg):,}")

# Join a la grilla base: celdas sin siniestro -> 0
matriz_a = grilla.merge(agg, on=["distrito_key", "fecha", "franja_horaria"], how="left")
for col in ["n_siniestros", "fallecidos", "lesionados"]:
    matriz_a[col] = matriz_a[col].fillna(0).astype(int)

print(f"Matriz A tras join: {matriz_a.shape[0]:,} filas × {matriz_a.shape[1]} columnas")
assert matriz_a.shape[0] == grilla.shape[0], "el join alteró el número de celdas"

Celdas con al menos 1 siniestro: 69,593
Matriz A tras join: 676,504 filas × 6 columnas


### 4.2 Agregados de severidad por celda (insumo descriptivo)

Se agregan también `fallecidos` y `lesionados` por celda. **No son features de la
Matriz A** (son variables ex-post: solo se conocen una vez ocurrido el siniestro, y
usarlas para predecir frecuencia sería fuga de información). Se calculan aquí porque:

- Permiten construir una tasa de letalidad por zona como capa **descriptiva** para la
  visualización de Etapa 2 (ST-DBSCAN / mapas de riesgo).
- Sirven de control de integridad: la suma de `fallecidos`/`lesionados` sobre la grilla
  debe coincidir con el total del dataset crudo (nada se pierde ni se duplica al agregar).


In [14]:


# --- 4.2 Control de integridad: nada se pierde ni se duplica al agregar ---
tot_grilla = {
    "siniestros": int(matriz_a["n_siniestros"].sum()),
    "fallecidos": int(matriz_a["fallecidos"].sum()),
    "lesionados": int(matriz_a["lesionados"].sum()),
}
tot_crudo = {
    "siniestros": len(siniestros),
    "fallecidos": int(siniestros["fallecidos"].sum()),
    "lesionados": int(siniestros["lesionados"].sum()),
}

print("\nCONTROL DE INTEGRIDAD (grilla agregada vs crudo)")
print("-" * 50)
for k in tot_crudo:
    ok = tot_grilla[k] == tot_crudo[k]
    print(f"  {k:<11}: grilla={tot_grilla[k]:>7,}  crudo={tot_crudo[k]:>7,}  {'OK' if ok else 'DIFIERE'}")
    assert ok, f"total de {k} no coincide tras la agregación"
print(">> Integridad OK: la agregación conserva todos los conteos.")




CONTROL DE INTEGRIDAD (grilla agregada vs crudo)
--------------------------------------------------
  siniestros : grilla= 89,946  crudo= 89,946  OK
  fallecidos : grilla=  5,883  crudo=  5,883  OK
  lesionados : grilla= 54,519  crudo= 54,519  OK
>> Integridad OK: la agregación conserva todos los conteos.



### 4.3 Diagnóstico de zero-inflation

Tras la agregación se reporta la proporción de celdas en cero y la media de siniestros
por celda. Se anticipa una **altísima proporción de ceros** (la unidad es muy granular:
un distrito pequeño, un día, una franja de 6 horas). Este diagnóstico confirma la
naturaleza del problema de conteo y refuerza la elección de modelos de la familia
Poisson/Binomial Negativa sobre una regresión lineal ordinaria.

In [15]:
# --- 4.3 Diagnóstico de zero-inflation y sobredispersión ---
n_cero = int((matriz_a["n_siniestros"] == 0).sum())
pct_cero = 100 * n_cero / len(matriz_a)
media = matriz_a["n_siniestros"].mean()
var = matriz_a["n_siniestros"].var()

print("\nDISTRIBUCIÓN DE n_siniestros POR CELDA")
print("-" * 50)
print(f"  Celdas totales        : {len(matriz_a):,}")
print(f"  Celdas en cero        : {n_cero:,}  ({pct_cero:.1f}%)")
print(f"  Media                 : {media:.4f}")
print(f"  Varianza              : {var:.4f}")
print(f"  Índice de dispersión  : {var / media:.2f}  (var/media; >1 = sobredispersión)")
print(f"  Máximo en una celda   : {int(matriz_a['n_siniestros'].max())}")
print("\n  Conteo de valores (top):")
print(matriz_a["n_siniestros"].value_counts().sort_index().head(8).to_string())


DISTRIBUCIÓN DE n_siniestros POR CELDA
--------------------------------------------------
  Celdas totales        : 676,504
  Celdas en cero        : 606,911  (89.7%)
  Media                 : 0.1330
  Varianza              : 0.2048
  Índice de dispersión  : 1.54  (var/media; >1 = sobredispersión)
  Máximo en una celda   : 10

  Conteo de valores (top):
n_siniestros
0    606911
1     55829
2      9415
3      2856
4       990
5       340
6       107
7        38


In [17]:
# Ver TODOS los valores de tipo y cuántas filas hay de cada uno
print("Conteo por tipo:")
print(feriados["tipo"].value_counts().to_string())

# Ver los feriados que NO son Nacional (todos los locales, sea cual sea el distrito)
print("\nFeriados no-Nacional:")
mask_local = ~feriados["tipo"].str.strip().str.lower().str.startswith("nacional")
print(feriados[mask_local][["anio", "fecha", "nombre_feriado", "tipo"]].to_string())

Conteo por tipo:
tipo
Nacional                55
Local (San Salvador)    15

Feriados no-Nacional:
    anio      fecha                    nombre_feriado                  tipo
7   2022 2022-08-03  Fiestas Agostinas (San Salvador)  Local (San Salvador)
8   2022 2022-08-04  Fiestas Agostinas (San Salvador)  Local (San Salvador)
9   2022 2022-08-05  Fiestas Agostinas (San Salvador)  Local (San Salvador)
21  2023 2023-08-03  Fiestas Agostinas (San Salvador)  Local (San Salvador)
22  2023 2023-08-04  Fiestas Agostinas (San Salvador)  Local (San Salvador)
23  2023 2023-08-05  Fiestas Agostinas (San Salvador)  Local (San Salvador)
35  2024 2024-08-03  Fiestas Agostinas (San Salvador)  Local (San Salvador)
36  2024 2024-08-04  Fiestas Agostinas (San Salvador)  Local (San Salvador)
37  2024 2024-08-05  Fiestas Agostinas (San Salvador)  Local (San Salvador)
49  2025 2025-08-03  Fiestas Agostinas (San Salvador)  Local (San Salvador)
50  2025 2025-08-04  Fiestas Agostinas (San Salvador)  Local (San

## 5. Feature engineering — Calendario

Se derivan variables de calendario a partir de la columna `fecha` de la grilla. Todas son
**ex-ante** (conocidas antes de que ocurra cualquier siniestro), por lo que son features
válidas para la Matriz A (frecuencia). Estas mismas variables se reutilizan en la Matriz B.

### 5.1 Variables de calendario derivadas de la fecha

Este bloque construye las variables que dependen **solo de la fecha** (no requieren cruzar
con el archivo de feriados, que se trata aparte en 5.2 por su lógica nacional/local):

| Variable | Definición | Justificación (NB02) |
|---|---|---|
| `mes` | mes del año (1–12) | estacionalidad |
| `es_lluviosa` | 1 si mes ∈ May–Oct | +16.4% de siniestros en temporada lluviosa |
| `dia_semana` | 0=lunes … 6=domingo | patrón semanal |
| `es_finde` | 1 si día ∈ {viernes, sábado} | Sáb > Vie > Lun en el EDA |
| `periodo_agostino` | 1 si fecha ∈ 3–6 de agosto | pico nacional de movilidad agostina |

**Sobre `periodo_agostino`.** Se define como la ventana **3–6 de agosto**, aplicada a
**todo el país** (todos los distritos). Esta ventana está respaldada por dos evidencias
independientes: (1) la estructura legal de las Fiestas Agostinas —asueto local en San
Salvador los días 3 y 5, y asueto nacional el 6 (Código de Trabajo, art. 190)—, y (2) el
pico empírico observado en el propio dataset, donde los siniestros diarios del 3 al 6 de
agosto (≈310–360/día) superan claramente la base de días contiguos (≈210–225/día). A
diferencia del asueto legal —que es parcialmente local— la *movilidad* agostina es un
fenómeno nacional, por lo que esta variable se marca para todos los distritos.

**Sobre `es_finde` = viernes y sábado.** No es el fin de semana calendario (sáb–dom):
NB02 mostró el orden Sábado > Viernes > Lunes en siniestralidad, con el domingo más bajo.
Se codifica el par viernes–sábado como los días de mayor exposición nocturna/recreativa.

In [19]:
# =============================================================
# Sección 5.1 — Variables de calendario derivadas de la fecha
# Solo dependen de matriz_a["fecha"] (los feriados se tratan en 5.2).
# =============================================================

f = matriz_a["fecha"]

# mes (1–12)
matriz_a["mes"] = f.dt.month

# temporada lluviosa: mayo–octubre (May–Oct)
matriz_a["es_lluviosa"] = f.dt.month.between(5, 10).astype(int)

# día de la semana (0=lunes … 6=domingo)
matriz_a["dia_semana"] = f.dt.dayofweek

# fin de semana "de riesgo": viernes (4) y sábado (5), según orden de NB02
matriz_a["es_finde"] = f.dt.dayofweek.isin([4, 5]).astype(int)

# período agostino: 3–6 de agosto, nacional (todas las celdas)
matriz_a["periodo_agostino"] = (
    (f.dt.month == 8) & (f.dt.day.between(3, 6))
).astype(int)

# --- Verificación rápida ---
print("VARIABLES DE CALENDARIO (5.1)")
print("-" * 50)
print(f"  mes                : {sorted(matriz_a['mes'].unique())}")
print(f"  es_lluviosa (May-Oct): {matriz_a['es_lluviosa'].mean()*100:.1f}% de celdas")
print(f"  es_finde (Vie-Sáb)   : {matriz_a['es_finde'].mean()*100:.1f}% de celdas")
print(f"  periodo_agostino     : {int(matriz_a['periodo_agostino'].sum()):,} celdas "
      f"({matriz_a['periodo_agostino'].mean()*100:.2f}%)")

# Sanity: es_finde debe rondar 2/7 ≈ 28.6%; es_lluviosa 6/12 = 50% (aprox por días reales)
print("\n  Chequeo día_semana vs es_finde (filas de muestra):")
print(matriz_a[["fecha", "dia_semana", "es_finde", "es_lluviosa", "periodo_agostino"]]
      .drop_duplicates("fecha").head(8).to_string(index=False))

VARIABLES DE CALENDARIO (5.1)
--------------------------------------------------
  mes                : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  es_lluviosa (May-Oct): 48.5% de celdas
  es_finde (Vie-Sáb)   : 28.6% de celdas
  periodo_agostino     : 6,592 celdas (0.97%)

  Chequeo día_semana vs es_finde (filas de muestra):
     fecha  dia_semana  es_finde  es_lluviosa  periodo_agostino
2022-01-01           5         1            0                 0
2022-01-02           6         0            0                 0
2022-01-03           0         0            0                 0
2022-01-04           1         0            0                 0
2022-01-05           2         0            0                 0
2022-01-06           3         0            0                 0
2022-01-07           4         1            0                 0
2022-01-08           5         1            0                 0


### 5.2 Feriados: nacionales vs locales

El archivo de feriados distingue el alcance en la columna `tipo`. El valor no es un
`Local` genérico, sino que **embebe el distrito** entre paréntesis: los valores reales son
`Nacional` y `Local (San Salvador)`. Esta distinción es crítica para no inflar falsamente
el efecto de feriado en distritos donde el día fue laborable:

- **Feriado Nacional** → aplica a **todos** los distritos en esa fecha.
- **Feriado Local (San Salvador)** → aplica **solo al distrito San Salvador**. En el
  dataset son las Fiestas Agostinas (3, 4 y 5 de agosto), asueto de ley circunscrito a la
  capital (Código de Trabajo, art. 190).

**Construcción de `es_feriado`.** Se arma en dos pasos y se combinan:

1. **Componente nacional:** una celda tiene `es_feriado = 1` si su `fecha` coincide con
   cualquier feriado `Nacional`, sin importar el distrito.
2. **Componente local:** una celda tiene `es_feriado = 1` si su `fecha` coincide con un
   feriado `Local (San Salvador)` **y** su `distrito_key` es `san salvador`.

**Fidelidad a la fuente (Ruta 1b).** `es_feriado` se construye exactamente como lo declara
el archivo de feriados, sin agregar ni quitar fechas. El asueto nacional del 6 de agosto
—que la ley reconoce pero el archivo no lista— **no** se inyecta aquí; su efecto de
movilidad ya queda capturado por `periodo_agostino` (ventana 3–6, nacional, definida en
5.1). Así se evita contradecir la fuente y no se pierde la señal del 6.

**Limitación documentada.** La ley concede además un día de asueto por la fiesta patronal
propia de cada distrito, que varía por localidad. No se dispone del calendario patronal de
los 103 distritos, por lo que ese asueto local distrito-específico **no se modela**. Se
declara como limitación y línea futura.

In [21]:
# =============================================================
# Sección 5.2 — Feriados nacionales vs locales -> es_feriado
# Nacional: aplica a todos los distritos en esa fecha.
# Local (San Salvador): aplica SOLO al distrito 'san salvador'.
# =============================================================

# Normalizar la fecha del archivo de feriados a nivel día
feriados = feriados.copy()
feriados["_fecha_dia"] = pd.to_datetime(feriados["fecha"]).dt.normalize()

# Clasificar cada feriado por alcance según 'tipo'
tipo_norm = feriados["tipo"].str.strip().str.lower()
es_nacional = tipo_norm.str.startswith("nacional")
es_local_ss = tipo_norm.str.startswith("local")  # en este dataset: solo San Salvador

fechas_nacionales = set(feriados.loc[es_nacional, "_fecha_dia"])
fechas_local_ss   = set(feriados.loc[es_local_ss, "_fecha_dia"])

print(f"Fechas feriado Nacional          : {len(fechas_nacionales)}")
print(f"Fechas feriado Local (San Salv.) : {len(fechas_local_ss)}")

# --- Componente nacional: cualquier celda cuya fecha sea feriado nacional ---
comp_nacional = matriz_a["fecha"].isin(fechas_nacionales)

# --- Componente local: fecha es local SS Y el distrito es san salvador ---
comp_local = (
    matriz_a["fecha"].isin(fechas_local_ss)
    & (matriz_a["distrito_key"] == "san salvador")
)

matriz_a["es_feriado"] = (comp_nacional | comp_local).astype(int)

# --- Verificación ---
print("\nCONSTRUCCIÓN DE es_feriado")
print("-" * 50)
print(f"  Celdas feriado (total)      : {int(matriz_a['es_feriado'].sum()):,}")
print(f"  Por componente nacional     : {int(comp_nacional.sum()):,}")
print(f"  Por componente local (SS)   : {int(comp_local.sum()):,}")

# Sanity: el componente local solo puede afectar al distrito san salvador
distritos_local = matriz_a.loc[comp_local, "distrito_key"].unique()
print(f"  Distritos afectados por local: {list(distritos_local)}")
assert set(distritos_local).issubset({"san salvador"}) or len(distritos_local) == 0, \
    "el feriado local afectó distritos distintos de san salvador"
print("  >> OK: el feriado local solo afecta al distrito san salvador.")

# --- Evidencia de la asimetría de la fuente de feriados (limitación conocida) ---
# El archivo solo lista feriados locales de San Salvador; NO incluye las fiestas
# patronales propias de los otros 102 distritos (asueto local de ley, art. 190).
# Consecuencia: es_feriado subrepresenta el efecto feriado-local fuera de la capital.
distritos_con_local = matriz_a.loc[
    matriz_a["es_feriado"].eq(1) & matriz_a["fecha"].isin(fechas_local_ss),
    "distrito_key",
].nunique()
print("\nASIMETRÍA DE LA FUENTE DE FERIADOS (limitación documentada)")
print("-" * 50)
print(f"  Distritos con feriado LOCAL en la fuente : {distritos_con_local} de 103")
print(f"  >> Los otros {103 - distritos_con_local} distritos NO tienen sus fiestas")
print(f"     patronales en el archivo. Limitación de datos, no del modelo.")

Fechas feriado Nacional          : 55
Fechas feriado Local (San Salv.) : 15

CONSTRUCCIÓN DE es_feriado
--------------------------------------------------
  Celdas feriado (total)      : 21,060
  Por componente nacional     : 21,012
  Por componente local (SS)   : 48
  Distritos afectados por local: ['san salvador']
  >> OK: el feriado local solo afecta al distrito san salvador.

ASIMETRÍA DE LA FUENTE DE FERIADOS (limitación documentada)
--------------------------------------------------
  Distritos con feriado LOCAL en la fuente : 1 de 103
  >> Los otros 102 distritos NO tienen sus fiestas
     patronales en el archivo. Limitación de datos, no del modelo.


### 5.3 Nota de colinealidad: `mes` vs `es_lluviosa`

Las variables `mes` y `es_lluviosa` no son independientes **por construcción**:
`es_lluviosa` se deriva directamente de `mes` (es 1 exactamente cuando el mes cae en
mayo–octubre). NB02 cuantificó esta relación en una correlación de r ≈ 0.32, y la
dependencia es estructural, no casual.

Incluir ambas simultáneamente en un modelo introduciría **colinealidad**: información
redundante que infla la varianza de los coeficientes y dificulta interpretar el efecto
individual de cada variable —justamente lo que se busca evitar en un modelo de conteo
interpretable (Poisson / Binomial Negativa).

**Ambas se construyen en esta sección**, porque cada una podría ser preferible según el
modelo:

- `es_lluviosa` es más **parsimoniosa** e interpretable (una sola variable binaria que
  captura la estacionalidad climática, alineada con el hallazgo de NB02 de +16.4% en
  temporada lluviosa).
- `mes` es más **granular** (captura variaciones intra-temporada que una binaria no puede)
  pero, al ser categórica de 12 niveles, aporta más parámetros y menos interpretabilidad
  directa.

**La decisión de cuál conservar NO se toma aquí.** Se difiere a la **Sección 11
(colinealidad y selección de variables)**, donde se evalúan todas las features juntas
mediante matriz de correlación y VIF, y se documenta la elección final con criterio
cuantitativo. Esta nota deja constancia de que la redundancia es **conocida y deliberada**
en esta etapa: se construyen ambas para decidir con evidencia, no se incluirán ambas en el
modelo final.

In [22]:
# Distribución de población en los 103 distritos observados (para fijar el umbral del flag)
pob_obs = censo[censo.set_index(["departamento_key","distrito_key"]).index.isin(
    set(zip(matriz_a["distrito_key"].map(lambda x: x), matriz_a["distrito_key"]))
)] if False else censo.merge(
    matriz_a[["distrito_key"]].drop_duplicates(), on="distrito_key", how="inner"
)
print(f"Distritos observados con población: {pob_obs['distrito_key'].nunique()}")
print("\nEstadísticos de población (103 distritos):")
print(pob_obs["poblacion"].describe().to_string())
print("\n10 distritos de MENOR población:")
print(pob_obs.nsmallest(10, "poblacion")[["distrito_key","departamento_key","poblacion"]].to_string(index=False))
print("\nPercentiles bajos:")
for p in [1, 5, 10, 25]:
    print(f"  p{p:>2}: {pob_obs['poblacion'].quantile(p/100):,.0f}")

Distritos observados con población: 103

Estadísticos de población (103 distritos):
count       103.000000
mean      46148.300971
std       53796.871891
min        2096.000000
25%       16680.000000
50%       27587.000000
75%       55474.000000
max      330543.000000

10 distritos de MENOR población:
 distrito_key departamento_key  poblacion
     cinquera          cabanas       2096
      perquin          morazan       2564
villa dolores          cabanas       6267
santo domingo      san vicente       6467
   tenancingo        cuscatlan       6707
     intipuca         la union       6785
      quelepa       san miguel       7062
agua caliente     chalatenango       7341
      osicala          morazan       9177
       jocoro          morazan       9658

Percentiles bajos:
  p 1: 2,638
  p 5: 6,813
  p10: 9,882
  p25: 16,680


## 6. Feature engineering — Exposición

Un conteo crudo de siniestros no es comparable entre distritos de tamaño muy distinto:
San Salvador tendrá más accidentes que un distrito rural simplemente porque tiene más
gente y más tráfico, no porque sea intrínsecamente más peligroso. Para que el modelo de
frecuencia sea interpretable, se necesita un **denominador de exposición** que normalice
el riesgo.

### 6.1 Denominador de exposición y offset del modelo

Se usa la **población residente por distrito** (VII Censo 2024, BCR/ONEC) como medida de
exposición. Es un denominador real, verificable y oficial, disponible en `data/raw/`.

**Por qué solo población y no parque vehicular.** El diseño ideal de exposición sumaría el
parque vehicular por distrito (número de vehículos circulantes). Sin embargo, **no existe
un archivo de parque vehicular por distrito en las fuentes disponibles** del proyecto. Se
adopta población como denominador —el estándar mínimo aceptado en análisis de seguridad
vial para tasas per cápita— y el parque vehicular se declara como **línea futura**.

**Sesgo conocido del denominador (documentado, no oculto).** La población *residente*
subestima la exposición real en distritos que reciben **commuting** diario —principalmente
del AMSS (San Salvador, Antiguo Cuscatlán)—, donde circulan muchas más personas de las que
residen. Esto puede inflar artificialmente la *tasa* de siniestralidad de esos distritos.
El propio anteproyecto ya reconoce esta limitante. Se declara explícitamente y se propone
como mejora futura (incorporar matrices origen-destino o parque vehicular).

**Uso en el modelo: offset, no feature.** La exposición **no entra como una variable
predictora más**. En un modelo de conteo (Poisson / Binomial Negativa) se incorpora como
**offset**: el término `log(exposición)` con coeficiente fijo en 1. Esto convierte el
modelo de "predecir número de siniestros" a "predecir la *tasa* de siniestros por unidad
de población", que es lo correcto metodológicamente. Aquí se construye la columna
`log_exposicion`, lista para ese uso en NB04.

In [25]:
# =============================================================
# Sección 6.1 — Denominador de exposición (población Censo 2024) y offset
# Paso 1: agregar departamento_key a la grilla (la grilla de la Sección 3 solo
#         traía distrito_key; se deriva el departamento vía mapa 1:1 desde siniestros).
# Paso 2: join de población por clave compuesta (departamento_key, distrito_key).
# Paso 3: construir log_exposicion para usar como offset en NB04 (Poisson/NB).
# =============================================================
import numpy as np

# --- Paso 1: agregar departamento_key a la grilla ---
# distrito_key es único entre los 103 observados -> mapa distrito->depto es 1:1
mapa_depto = (
    siniestros[["distrito_key", "departamento_key"]]
    .drop_duplicates()
    .set_index("distrito_key")["departamento_key"]
)
assert mapa_depto.index.is_unique, "distrito_key no es único: revisar homónimos"
matriz_a["departamento_key"] = matriz_a["distrito_key"].map(mapa_depto)

sin_depto = int(matriz_a["departamento_key"].isna().sum())
assert sin_depto == 0, "hay celdas sin departamento_key (revisar mapa distrito->depto)"
print(f"Paso 1 — departamento_key agregado a la grilla (celdas sin depto: {sin_depto})")

# --- Paso 2: join de población por clave compuesta ---
pob = censo[["departamento_key", "distrito_key", "poblacion", "pct_urbano"]].copy()

antes = matriz_a.shape[0]
matriz_a = matriz_a.merge(pob, on=["departamento_key", "distrito_key"], how="left")

print("\nJOIN DE EXPOSICIÓN (población Censo 2024)")
print("-" * 50)
print(f"  Filas antes del join : {antes:,}")
print(f"  Filas después        : {matriz_a.shape[0]:,}")
assert matriz_a.shape[0] == antes, "el join de población alteró el número de filas"

sin_pob = int(matriz_a["poblacion"].isna().sum())
print(f"  Celdas sin población : {sin_pob}")
assert sin_pob == 0, "hay celdas sin población tras el join (revisar clave de distrito)"

# --- Paso 3: offset = log de la exposición (población) ---
matriz_a["log_exposicion"] = np.log(matriz_a["poblacion"])

print(f"\n  Población — min: {matriz_a['poblacion'].min():,}  "
      f"max: {matriz_a['poblacion'].max():,}")
print(f"  log_exposicion — min: {matriz_a['log_exposicion'].min():.3f}  "
      f"max: {matriz_a['log_exposicion'].max():.3f}")
print("  >> OK: exposición unida y log_exposicion lista para offset en NB04.")

Paso 1 — departamento_key agregado a la grilla (celdas sin depto: 0)

JOIN DE EXPOSICIÓN (población Censo 2024)
--------------------------------------------------
  Filas antes del join : 676,504
  Filas después        : 676,504
  Celdas sin población : 0

  Población — min: 2,096  max: 330,543
  log_exposicion — min: 7.648  max: 12.708
  >> OK: exposición unida y log_exposicion lista para offset en NB04.


### 6.2 Flag de distritos de población muy baja

Los distritos con población muy pequeña producen **tasas inestables**: un solo siniestro
sobre una población de pocos miles dispara la tasa per cápita, generando valores atípicos
que no reflejan un riesgo estructural sino la aritmética de un denominador pequeño.

Se añade un flag `poblacion_baja` que marca los distritos con **población menor a 10,000
habitantes**. Este umbral es interpretable ("distritos de menos de diez mil personas") y
coincide con el corte natural de la distribución (≈ percentil 10 de los 103 distritos
observados), por lo que no es arbitrario. Marca la cola genuinamente pequeña (Cinquera,
Perquín, Villa Dolores, y similares).

**El flag no elimina distritos.** Solo los señala, de modo que en el análisis y el
modelado se pueda decidir explícitamente si filtrarlos, tratarlos aparte o conservarlos,
sin removerlos silenciosamente de los datos. La decisión de qué hacer con ellos se toma de
forma informada, no por omisión.

In [26]:
# =============================================================
# Sección 6.2 — Flag de distritos de población muy baja
# Marca (no elimina) distritos con población < 10,000 habitantes.
# =============================================================
UMBRAL_POB_BAJA = 10_000

matriz_a["poblacion_baja"] = (matriz_a["poblacion"] < UMBRAL_POB_BAJA).astype(int)

# --- Verificación: qué distritos quedan marcados ---
marcados = (
    matriz_a.loc[matriz_a["poblacion_baja"].eq(1),
                 ["distrito_key", "departamento_key", "poblacion"]]
    .drop_duplicates()
    .sort_values("poblacion")
)
n_dist_marcados = marcados["distrito_key"].nunique()
pct_celdas = 100 * matriz_a["poblacion_baja"].mean()

print(f"FLAG poblacion_baja (< {UMBRAL_POB_BAJA:,} hab.)")
print("-" * 50)
print(f"  Distritos marcados : {n_dist_marcados} de 103")
print(f"  Celdas marcadas    : {int(matriz_a['poblacion_baja'].sum()):,} ({pct_celdas:.1f}%)")
print("\n  Distritos con población baja:")
print(marcados.to_string(index=False))

FLAG poblacion_baja (< 10,000 hab.)
--------------------------------------------------
  Distritos marcados : 12 de 103
  Celdas marcadas    : 78,816 (11.7%)

  Distritos con población baja:
 distrito_key departamento_key  poblacion
     cinquera          cabanas       2096
      perquin          morazan       2564
villa dolores          cabanas       6267
santo domingo      san vicente       6467
   tenancingo        cuscatlan       6707
     intipuca         la union       6785
      quelepa       san miguel       7062
agua caliente     chalatenango       7341
      osicala          morazan       9177
       jocoro          morazan       9658
   teotepeque      la libertad       9862
     sociedad          morazan       9963


In [28]:
# ¿Siniestros tiene coordenadas GPS? ¿Con qué nombres?
geo = [c for c in siniestros.columns if any(k in c.lower() for k in ["lat", "lon", "lng", "coord", "gps"])]
print("Columnas geográficas en siniestros:", geo)
if geo:
    print("\nMuestra:")
    print(siniestros[geo].describe().to_string())
    print(f"\nNulos por columna:")
    print(siniestros[geo].isna().sum().to_string())

Columnas geográficas en siniestros: ['latitud', 'longitud']

Muestra:
            latitud      longitud
count  89946.000000  89946.000000
mean      13.701856    -89.054660
std        0.184581      0.495990
min       13.150633    -90.088157
25%       13.620248    -89.308055
50%       13.702979    -89.199664
75%       13.779472    -88.883399
max       14.404197    -87.772431

Nulos por columna:
latitud     0
longitud    0


## 7. Feature engineering — Clima estacional

NB02 estableció un hallazgo central sobre el clima: la señal de precipitación vive en
**resolución estacional/mensual**, no diaria. La correlación entre lluvia y siniestros es
fuerte a nivel mensual (r ≈ 0.814) pero prácticamente nula a nivel distrito×día
(r ≈ 0.006). La interpretación no es que la lluvia no importe, sino que el **instrumento**
correcto es la agregación estacional: el día puntual de lluvia en un distrito es ruido; el
patrón mensual de un régimen climático es señal.

En consecuencia, la feature de clima de la Matriz A se construye a **resolución mensual por
estación**, no como precipitación diaria puntual. Esto además evita inflar artificialmente
la dimensionalidad con una variable diaria de baja señal.

### 7.1 Asignación de cada distrito a su estación más cercana

Como no existe una estación meteorológica por distrito (solo hay 6 estaciones para todo el
país), cada distrito se asigna a la **estación más cercana**, replicando la lógica de NB02.

**Ubicación de cada distrito.** El censo no incluye coordenadas de distrito. Se deriva un
**centroide** por distrito promediando la latitud y longitud de todos sus siniestros
registrados (89,946 coordenadas completas, sin nulos). Es una aproximación al centro de
actividad vial del distrito, adecuada para asignar la estación de referencia.

**Estaciones usables (decisión de NB02).** De las 6 estaciones con datos, se **excluye la
estación SS** (San Salvador centro, 78662): NB02 documentó que su registro de precipitación
tiene solo 22.7% de completitud. Su vecina IL (Ilopango, 78663), a ~8 km y en la misma
zona del AMSS, tiene prcp al 99%, por lo que absorbe los distritos de la capital sin pérdida
de cobertura. Quedan **5 estaciones usables**: IL, SA, SO, PAZ, SM.

**Distancia.** Se usa la distancia euclidiana sobre coordenadas geográficas. Dada la
pequeña extensión de El Salvador (~21,000 km²) y que solo se busca la estación *más
cercana* (un ranking relativo, no una medida absoluta de km), la aproximación euclidiana
es suficiente; no se requiere distancia de Haversine para este propósito.

In [29]:
# =============================================================
# Sección 7.1 — Asignación distrito -> estación más cercana
# Centroide por distrito (promedio de coords de sus siniestros) vs estaciones
# usables (excluye SS por decisión NB02). Distancia euclidiana, más cercana gana.
# =============================================================

# --- Centroide de cada distrito (promedio de lat/lon de sus siniestros) ---
centroides = (
    siniestros.groupby("distrito_key")[["latitud", "longitud"]]
    .mean()
    .reset_index()
    .rename(columns={"latitud": "dist_lat", "longitud": "dist_lon"})
)
print(f"Centroides calculados para {len(centroides)} distritos")

# --- Estaciones usables: excluir SS (decisión NB02: prcp 22.7%) ---
est = cfg.estaciones_con_datos()
est_usables = est[est["clima_key"] != "SS"].reset_index(drop=True)
print(f"Estaciones usables (excluye SS): {list(est_usables['clima_key'])}")

# --- Asignar a cada distrito la estación más cercana (euclidiana) ---
def _estacion_mas_cercana(lat, lon):
    d2 = (est_usables["latitud"] - lat) ** 2 + (est_usables["longitud"] - lon) ** 2
    return est_usables.loc[d2.idxmin(), "clima_key"]

centroides["estacion_asignada"] = centroides.apply(
    lambda r: _estacion_mas_cercana(r["dist_lat"], r["dist_lon"]), axis=1
)

# --- Verificación: distribución de distritos por estación ---
print("\nDISTRITOS ASIGNADOS POR ESTACIÓN")
print("-" * 50)
print(centroides["estacion_asignada"].value_counts().to_string())

# Chequeo puntual: San Salvador debe caer en IL (Ilopango), no en SS
ss_row = centroides[centroides["distrito_key"] == "san salvador"]
if not ss_row.empty:
    print(f"\n  Distrito 'san salvador' -> estación: "
          f"{ss_row['estacion_asignada'].iloc[0]}  (esperado: IL)")

Centroides calculados para 103 distritos
Estaciones usables (excluye SS): ['IL', 'SA', 'SO', 'PAZ', 'SM']

DISTRITOS ASIGNADOS POR ESTACIÓN
--------------------------------------------------
estacion_asignada
IL     41
SM     27
SA     16
SO     11
PAZ     8

  Distrito 'san salvador' -> estación: IL  (esperado: IL)


### 7.2 Agregación mensual de precipitación y join a la grilla

Coherente con el hallazgo de NB02 (la señal climática vive en resolución mensual, no
diaria), la precipitación se agrega a **total mensual por estación** y se une a cada celda
de la grilla según el **mes y la estación asignada** a su distrito.

**Qué representa `prcp_mensual`.** Para una celda cualquiera —p. ej. San Salvador, 15 de
marzo 2023, franja Mañana— el valor de `prcp_mensual` es la **lluvia total caída en marzo
2023** en la estación asignada a San Salvador (IL/Ilopango). Todas las celdas del mismo
distrito y mes comparten ese valor: no es la lluvia de ese día ni de esa hora, sino el
régimen de lluvia del mes, que es lo que predice la frecuencia de siniestros.

**Por qué suma mensual y no diaria/horaria.** La resolución horaria del accidente se
agrega al construir la grilla (la Matriz A modela conteos por franja, no accidentes
individuales). La precipitación diaria por distrito mostró correlación nula con los
siniestros (r ≈ 0.006 en NB02), dominada por ruido; la agregación mensual recupera la
señal estacional (r ≈ 0.814). La lógica de "condición del momento" pertenece a la Matriz B
(severidad), donde ya está representada por `condicion_via`.

**Pasos:**
1. Filtrar `clima` a las 5 estaciones usables (sin SS) y agregar `prcp` por estación y mes.
2. Unir cada distrito con su estación asignada (Sección 7.1).
3. Unir a la grilla por `(estación, año-mes)` → cada celda hereda la lluvia mensual de su
   estación.

**Nota de cobertura temporal.** El clima llega hasta jul-2026 y la grilla corta en
jun-2026; el join por mes solo toma los meses presentes en la grilla, sin arrastrar meses
sobrantes. Se verifica que ninguna celda quede sin `prcp_mensual`.

In [30]:
# =============================================================
# Sección 7.2 — Precipitación mensual por estación -> join a la grilla
# Suma de prcp por estación y mes; cada celda hereda la lluvia mensual
# de la estación asignada a su distrito (Sección 7.1).
# =============================================================

# --- Paso 1: agregar prcp por estación y año-mes (solo estaciones usables) ---
usables = list(est_usables["clima_key"])
clima_u = clima[clima["estacion"].isin(usables)].copy()
clima_u["anio_mes"] = clima_u["time"].dt.to_period("M")

# Completitud de prcp en las estaciones usables (control)
compl = clima_u.groupby("estacion")["prcp"].apply(lambda s: 100 * s.notna().mean())
print("Completitud de prcp por estación usable (%):")
print(compl.round(1).to_string())

prcp_mens = (
    clima_u.groupby(["estacion", "anio_mes"])["prcp"]
    .sum(min_count=1)              # suma mensual; min_count=1 -> NaN si el mes está vacío
    .reset_index()
    .rename(columns={"prcp": "prcp_mensual", "estacion": "estacion_asignada"})
)
print(f"\nFilas de precipitación mensual: {len(prcp_mens)} "
      f"(estaciones × meses)")

# --- Paso 2: mapa distrito -> estación asignada (de la Sección 7.1) ---
mapa_est = centroides.set_index("distrito_key")["estacion_asignada"]
matriz_a["estacion_asignada"] = matriz_a["distrito_key"].map(mapa_est)

# --- Paso 3: join a la grilla por (estación, año-mes) ---
matriz_a["anio_mes"] = matriz_a["fecha"].dt.to_period("M")
antes = matriz_a.shape[0]
matriz_a = matriz_a.merge(
    prcp_mens, on=["estacion_asignada", "anio_mes"], how="left"
)
print(f"\nJOIN DE CLIMA MENSUAL")
print("-" * 50)
print(f"  Filas antes : {antes:,}   después : {matriz_a.shape[0]:,}")
assert matriz_a.shape[0] == antes, "el join de clima alteró el número de filas"

# --- Verificación de cobertura ---
sin_prcp = int(matriz_a["prcp_mensual"].isna().sum())
print(f"  Celdas sin prcp_mensual : {sin_prcp}  ({100*sin_prcp/len(matriz_a):.2f}%)")
print(f"  prcp_mensual — min: {matriz_a['prcp_mensual'].min():.1f}  "
      f"max: {matriz_a['prcp_mensual'].max():.1f}  "
      f"media: {matriz_a['prcp_mensual'].mean():.1f}")

Completitud de prcp por estación usable (%):
estacion
IL     99.0
PAZ    90.1
SA     99.1
SM     99.5
SO     99.2

Filas de precipitación mensual: 260 (estaciones × meses)

JOIN DE CLIMA MENSUAL
--------------------------------------------------
  Filas antes : 676,504   después : 676,504
  Celdas sin prcp_mensual : 37024  (5.47%)
  prcp_mensual — min: 0.0  max: 1030.7  media: 141.3


In [31]:
# ¿Qué celdas quedaron sin prcp_mensual? Confirmar el patrón (estación × mes)
falta = matriz_a[matriz_a["prcp_mensual"].isna()]
print(f"Celdas sin prcp_mensual: {len(falta):,}")

print("\nPor estación asignada:")
print(falta["estacion_asignada"].value_counts().to_string())

print("\nPor año-mes (meses afectados):")
print(falta["anio_mes"].value_counts().sort_index().to_string())

# ¿Hasta qué mes llega realmente cada estación usable?
print("\nÚltimo año-mes con prcp por estación (en clima crudo usable):")
ultimo = (clima_u.dropna(subset=["prcp"])
          .groupby("estacion")["anio_mes"].max())
print(ultimo.to_string())

Celdas sin prcp_mensual: 37,024

Por estación asignada:
estacion_asignada
IL     20008
SA      7808
SO      5368
PAZ     3840

Por año-mes (meses afectados):
anio_mes
2022-01     992
2022-02     896
2022-03     992
2022-04     960
2026-03    8432
2026-04    8160
2026-05    8432
2026-06    8160
Freq: M

Último año-mes con prcp por estación (en clima crudo usable):
estacion
IL     2026-02
PAZ    2026-07
SA     2026-02
SM     2026-07
SO     2026-02


### 7.3 Imputación climatológica de precipitación faltante

El join de clima dejó **37,024 celdas (5.47%) sin `prcp_mensual`**, por dos huecos en las
series meteorológicas:

- **Cola de 2026:** las estaciones IL, SA y SO (formato bulk) cortan su registro en
  feb-2026, mientras la grilla llega a jun-2026. Faltan mar–jun 2026 para los distritos de
  esas estaciones (33,184 celdas).
- **Inicio de 2022:** la estación PAZ no tiene precipitación registrada en ene–abr 2022
  (3,840 celdas), un hueco interno de la serie.

**Método: imputación climatológica por (estación, mes-calendario).** Cada celda faltante
hereda el **promedio histórico de ese mismo mes calendario** en los demás años disponibles
de su estación. Por ejemplo, marzo-2026 en IL se rellena con el promedio de los marzos
2022–2025 en IL; enero-2022 en PAZ, con el promedio de los eneros de los años disponibles.

**Por qué es defendible.** La precipitación en El Salvador es fuertemente estacional (seca
nov–abr, lluviosa may–oct), por lo que el "mes típico" de una estación es un buen estimador
de un mes faltante. Además, la feature busca capturar el **régimen estacional** —no el valor
exacto de un mes puntual—, de modo que la imputación climatológica es coherente con el
propósito de la variable, no un parche arbitrario. Arrastrar el último valor (feb-2026,
mes seco) habría subestimado gravemente la temporada lluviosa entrante.

**Trazabilidad.** Se añade el flag `prcp_imputada` (1 = valor estimado, 0 = observado) para
distinguir los ~5.5% de celdas imputadas de las observadas. Esta imputación se declara como
**limitación** en el documento final.

In [32]:
# =============================================================
# Sección 7.3 — Imputación climatológica de prcp_mensual faltante
# Cada celda sin dato hereda el promedio de ese mes-calendario en los
# demás años disponibles de su estación. Se marca con flag prcp_imputada.
# =============================================================

# Flag: 1 donde falta (antes de imputar)
matriz_a["prcp_imputada"] = matriz_a["prcp_mensual"].isna().astype(int)

# --- Climatología por (estación, mes-calendario) desde el clima mensual observado ---
prcp_mens = prcp_mens.copy()
prcp_mens["mes_cal"] = prcp_mens["anio_mes"].dt.month
clim = (
    prcp_mens.dropna(subset=["prcp_mensual"])
    .groupby(["estacion_asignada", "mes_cal"])["prcp_mensual"]
    .mean()
    .rename("prcp_clim")
    .reset_index()
)

# --- Rellenar faltantes de la grilla con la climatología ---
matriz_a["mes_cal"] = matriz_a["fecha"].dt.month
matriz_a = matriz_a.merge(clim, on=["estacion_asignada", "mes_cal"], how="left")

falt_antes = int(matriz_a["prcp_mensual"].isna().sum())
matriz_a["prcp_mensual"] = matriz_a["prcp_mensual"].fillna(matriz_a["prcp_clim"])
falt_despues = int(matriz_a["prcp_mensual"].isna().sum())

# Limpiar columnas auxiliares
matriz_a = matriz_a.drop(columns=["prcp_clim", "mes_cal"])

# --- Verificación ---
print("IMPUTACIÓN CLIMATOLÓGICA DE prcp_mensual")
print("-" * 50)
print(f"  Celdas faltantes antes  : {falt_antes:,}")
print(f"  Celdas faltantes después: {falt_despues:,}")
print(f"  Celdas imputadas (flag) : {int(matriz_a['prcp_imputada'].sum()):,} "
      f"({100*matriz_a['prcp_imputada'].mean():.2f}%)")
assert falt_despues == 0, "quedaron celdas sin prcp_mensual tras la imputación"

print(f"\n  prcp_mensual final — min: {matriz_a['prcp_mensual'].min():.1f}  "
      f"max: {matriz_a['prcp_mensual'].max():.1f}  "
      f"media: {matriz_a['prcp_mensual'].mean():.1f}")

# Comparación observado vs imputado (sanity: medias en rango similar)
obs_media = matriz_a.loc[matriz_a["prcp_imputada"].eq(0), "prcp_mensual"].mean()
imp_media = matriz_a.loc[matriz_a["prcp_imputada"].eq(1), "prcp_mensual"].mean()
print(f"  Media prcp observada : {obs_media:.1f}")
print(f"  Media prcp imputada  : {imp_media:.1f}")

IMPUTACIÓN CLIMATOLÓGICA DE prcp_mensual
--------------------------------------------------
  Celdas faltantes antes  : 37,024
  Celdas faltantes después: 0
  Celdas imputadas (flag) : 37,024 (5.47%)

  prcp_mensual final — min: 0.0  max: 1030.7  media: 141.2
  Media prcp observada : 141.3
  Media prcp imputada  : 139.2


## 8. Feature engineering — Categóricas de siniestro (Matriz B)

A partir de aquí se construye la **Matriz B (severidad)**, cuya unidad de análisis es el
**siniestro individual** (no la celda de grilla de la Matriz A). En esta matriz las
variables **ex-post** son predictores legítimos: como se condiciona a que el siniestro ya
ocurrió, describir *cómo* ocurrió (causa, tipo, vehículo) es información válida para
estimar su gravedad. Esto es el modelo estándar de *injury severity* en seguridad vial y
no constituye fuga de información (a diferencia de la Matriz A, donde estas variables están
prohibidas).

### 8.1 Agrupación de la cola de `factor_causante`

`factor_causante` tiene 17 categorías con una **cola larga**: las primeras ~8 concentran
la mayoría de los casos, mientras varias categorías del final tienen frecuencias ínfimas
—"Mal Estado del Vehículo" (0.06%), "Enfermedad" (0.05%), "Carga Mal Acondicionada"
(0.03%), "Deslumbramiento" (0.01%)—. Categorías tan raras aportan poca señal y elevan el
riesgo de sobreajuste en el clasificador (un modelo puede memorizar casos casi únicos).

**Decisión.** Se agrupan en la categoría **"Otros"** (ya existente, 2.22%) todas las
categorías con **frecuencia menor al 1%** del total: Falla Mecánica, Mal Estado del
Vehículo, Enfermedad, Carga Mal Acondicionada y Deslumbramiento. El resultado conserva
~12 categorías con masa suficiente.

**`tipo_siniestro` y `tipo_vehiculo` NO se agrupan** (decisión informada, no omisión).
Sus categorías minoritarias —Volcamiento (1.84%) en tipo_siniestro, "Otro" (2.07%) en
tipo_vehiculo— aún tienen suficientes casos (>1,600) y son cualitativamente distintas y
relevantes para la severidad. Agruparlas perdería información útil sin ganar estabilidad.
El principio aplicado: agrupar solo cuando la baja masa genera ruido, no por defecto.

In [34]:
# =============================================================
# Sección 8 — Matriz B (severidad): base = siniestro individual
# 8.1 Agrupar en "Otros" las categorías de factor_causante con <1% del total.
#     tipo_siniestro y tipo_vehiculo se dejan sin agrupar (decisión informada).
# =============================================================

# --- Base de la Matriz B: un registro por siniestro (copia para no tocar 'siniestros') ---
matriz_b = siniestros.copy()

# --- 8.1 Agrupar cola de factor_causante (<1%) en "Otros" ---
UMBRAL_FACTOR = 0.01  # 1% del total

frac = matriz_b["factor_causante"].value_counts(normalize=True)
cats_cola = frac[frac < UMBRAL_FACTOR].index.tolist()

matriz_b["factor_causante_grp"] = matriz_b["factor_causante"].where(
    ~matriz_b["factor_causante"].isin(cats_cola), "Otros"
)

print("AGRUPACIÓN DE factor_causante (<1% -> 'Otros')")
print("-" * 55)
print(f"  Categorías originales : {matriz_b['factor_causante'].nunique()}")
print(f"  Categorías agrupadas  : {matriz_b['factor_causante_grp'].nunique()}")
print(f"  Movidas a 'Otros'     : {cats_cola}")
print(f"\nDistribución final de factor_causante_grp:")
vc = matriz_b["factor_causante_grp"].value_counts()
print((pd.DataFrame({"n": vc, "pct": (100*vc/len(matriz_b)).round(2)})).to_string())

# --- tipo_siniestro y tipo_vehiculo: sin cambios (se documentan tal cual) ---
print(f"\ntipo_siniestro  : {matriz_b['tipo_siniestro'].nunique()} categorías (sin agrupar)")
print(f"tipo_vehiculo   : {matriz_b['tipo_vehiculo'].nunique()} categorías (sin agrupar)")

AGRUPACIÓN DE factor_causante (<1% -> 'Otros')
-------------------------------------------------------
  Categorías originales : 17
  Categorías agrupadas  : 12
  Movidas a 'Otros'     : ['Falla Mecánica', 'Mal Estado del Vehículo', 'Enfermedad', 'Carga Mal Acondicionada', 'Deslumbramiento']

Distribución final de factor_causante_grp:
                                        n    pct
factor_causante_grp                             
Distracción del Conductor           21428  23.82
Invadir Carril                      18412  20.47
No Respetar Señales de Tránsito     11796  13.11
No Guardar Distancia Reglamentaria  11552  12.84
Estado de Ebriedad o Droga           7263   8.07
Velocidad Excesiva                   5660   6.29
Circular en Reversa                  3887   4.32
Adelantamiento Antireglamentario     3138   3.49
Otros                                2954   3.28
Inexperiencia                        1544   1.72
Giro Incorrecto                      1262   1.40
Imprudencia del Peatón    

## 9. Feature engineering — Severidad: grupo vulnerable y rango etario

Se derivan dos atributos de la víctima que enriquecen la Matriz B: `grupo_vulnerable` y
`rango_etario`. Ambos provienen de `victimas`, que tiene **múltiples filas por siniestro**
(1.40 víctimas en promedio, hasta 7), por lo que deben **colapsarse a un valor único por
siniestro** antes de unirse a la Matriz B.

### 9.1 `grupo_vulnerable`

En seguridad vial, los **usuarios vulnerables** son quienes carecen de carrocería que los
proteja: motociclistas, peatones y ciclistas. Se deriva de `tipo_usuario` y `tipo_vehiculo`
de la víctima:

- **Peatón** → `tipo_usuario == "Peaton"`
- **Ciclista** → `tipo_vehiculo == "Bicicleta"`
- **Motociclista** → `tipo_vehiculo == "Motocicleta"`
- **Otro** → resto (ocupantes de automóvil, bus, etc.)

**Colapso por siniestro: el más vulnerable presente**, con prioridad
**peatón > ciclista > motociclista > otro**. Si un siniestro involucra un auto y un peatón,
se marca como "peatón": para la severidad, lo determinante es el peor escenario (el usuario
más expuesto involucrado), no un promedio.

### 9.2 `rango_etario`

Colapso: **rango etario del conductor** involucrado; si el siniestro no registra conductor
(p. ej. un atropello donde solo consta el peatón), se toma el de la primera víctima. El
conductor es el usuario más consistentemente presente y el más interpretable para analizar
la relación edad–severidad.

### 9.3 Uso: descriptivo, no predictor del clasificador de 3 clases

Ambas variables **solo existen cuando hay víctima**. En los ~46,841 siniestros de solo
daños materiales toman el valor `"Sin victimas"`. Por eso se guardan como **columnas
descriptivas/analíticas** en la Matriz B —útiles para el EDA de severidad (p. ej. el
hallazgo de NB02 de que los motociclistas son ~42% de los fatales) y para un sub-modelo
condicional lesionado-vs-fatal en Etapa 2—, pero **no se usan como predictores del
clasificador de 3 clases** (solo daños / lesionado / fatal): su ausencia coincidiría casi
perfectamente con la clase "solo daños", introduciendo fuga de información circular. Esta
restricción anti-leakage se documenta explícitamente.

In [36]:
# =============================================================
# Sección 9 — Severidad: grupo_vulnerable y rango_etario (Matriz B)
# Colapsa victimas (N por siniestro) a un valor único por siniestro.
# grupo_vulnerable = más vulnerable presente (peaton>ciclista>motociclista>otro)
# rango_etario = del conductor; si no hay, primera víctima.
# Ambas son columnas DESCRIPTIVAS (no features del clasificador de 3 clases).
# =============================================================

# --- 9.1 Clasificar cada víctima en su grupo vulnerable ---
def _grupo_victima(row):
    if row["tipo_usuario"] == "Peaton":
        return "Peaton"
    if row["tipo_vehiculo"] == "Bicicleta":
        return "Ciclista"
    if row["tipo_vehiculo"] == "Motocicleta":
        return "Motociclista"
    return "Otro"

vic = victimas.copy()
vic["_grupo"] = vic.apply(_grupo_victima, axis=1)

# Colapso por siniestro: el más vulnerable presente (orden de prioridad)
PRIORIDAD = {"Peaton": 0, "Ciclista": 1, "Motociclista": 2, "Otro": 3}
vic["_prio"] = vic["_grupo"].map(PRIORIDAD)
grupo_por_sin = (
    vic.sort_values("_prio")
    .groupby("id_siniestro")["_grupo"]
    .first()                       # el de menor _prio = más vulnerable
    .rename("grupo_vulnerable")
)

# --- 9.2 rango_etario del conductor; si no hay, primera víctima ---
def _rango_siniestro(g):
    cond = g[g["tipo_usuario"] == "Conductor"]
    if len(cond):
        return cond["rango_etario"].iloc[0]
    return g["rango_etario"].iloc[0]

rango_por_sin = (
    vic.groupby("id_siniestro", group_keys=False)
    .apply(_rango_siniestro)
    .rename("rango_etario")
)

# --- Unir a la Matriz B; solo-daños -> "Sin victimas" ---
matriz_b = matriz_b.merge(grupo_por_sin, on="id_siniestro", how="left")
matriz_b = matriz_b.merge(rango_por_sin, on="id_siniestro", how="left")
matriz_b["grupo_vulnerable"] = matriz_b["grupo_vulnerable"].fillna("Sin victimas")
matriz_b["rango_etario"] = matriz_b["rango_etario"].fillna("Sin victimas")

# --- Verificación ---
print("SEVERIDAD — grupo_vulnerable y rango_etario (Matriz B)")
print("-" * 55)
print(f"  Siniestros totales      : {len(matriz_b):,}")
print(f"  Con víctimas            : {int((matriz_b['grupo_vulnerable'] != 'Sin victimas').sum()):,}")
print(f"  Solo daños (Sin victimas): {int((matriz_b['grupo_vulnerable'] == 'Sin victimas').sum()):,}")

print(f"\ngrupo_vulnerable:")
print(matriz_b["grupo_vulnerable"].value_counts().to_string())

# Chequeo de consistencia: 'Sin victimas' debe cuadrar en ambas columnas
inconsist = int(((matriz_b["grupo_vulnerable"] == "Sin victimas") !=
                 (matriz_b["rango_etario"] == "Sin victimas")).sum())
print(f"\n  Inconsistencias 'Sin victimas' entre ambas columnas: {inconsist}")
assert inconsist == 0, "desalineación entre grupo_vulnerable y rango_etario en solo-daños"
print("  >> OK: columnas de severidad construidas y consistentes.")

SEVERIDAD — grupo_vulnerable y rango_etario (Matriz B)
-------------------------------------------------------
  Siniestros totales      : 89,946
  Con víctimas            : 43,105
  Solo daños (Sin victimas): 46,841

grupo_vulnerable:
grupo_vulnerable
Sin victimas    46841
Otro            14585
Motociclista    13907
Peaton          10455
Ciclista         4158

  Inconsistencias 'Sin victimas' entre ambas columnas: 0
  >> OK: columnas de severidad construidas y consistentes.


/var/folders/vr/117fzs5j7g36v1wtxyh0544w0000gp/T/ipykernel_29417/1650787442.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_rango_siniestro)


## 10. Definición de targets

El proyecto tiene dos targets, uno por matriz, correspondientes a los dos problemas de
modelado de NB04:

### 10.1 Matriz A — `n_siniestros` (regresión / conteo)

Ya construido en la Sección 4. Es el conteo de siniestros por celda
`distrito × fecha × franja`. Se modela con Regresión de Poisson y Binomial Negativa
(la sobredispersión y el 89.7% de ceros lo justifican), usando `log_exposicion` como
offset para predecir *tasa* en lugar de conteo crudo.

### 10.2 Matriz B — `nivel_riesgo` (clasificación, 3 clases)

Se construye a partir de la severidad observada del siniestro:

| Clase | Definición | Interpretación |
|---|---|---|
| `solo_danos` | `fallecidos == 0` y `lesionados == 0` | solo daños materiales |
| `con_lesionados` | `lesionados > 0` y `fallecidos == 0` | hay heridos, sin muertes |
| `fatal` | `fallecidos > 0` | al menos un fallecido |

Las clases son **mutuamente excluyentes y exhaustivas** (todo siniestro cae en una y solo
una). El orden refleja severidad creciente: daños < lesionados < fatal.

**Desbalance esperado.** La clase `fatal` es minoritaria (los siniestros mortales son una
fracción pequeña del total), y `solo_danos` es la mayoritaria. Este desbalance se
cuantifica aquí y se maneja en NB04 (p. ej. métricas por clase, F1 macro, class weights o
remuestreo), no se corrige en el preprocesamiento.

**Predictores admitidos (regla anti-leakage).** El clasificador de 3 clases usa como
features las **categóricas del siniestro** presentes en todos los registros
(`factor_causante_grp`, `tipo_siniestro`, `tipo_vehiculo`, `condicion_via`) más las
features ex-ante (calendario, clima, ubicación). Las variables de víctima
(`grupo_vulnerable`, `rango_etario`) **no** se usan como predictores de este target, por la
fuga circular explicada en la Sección 9.

In [38]:
# =============================================================
# Sección 10 — Definición de targets
# Matriz A: n_siniestros (ya construido en Sección 4) — regresión.
# Matriz B: nivel_riesgo (3 clases) — clasificación.
# =============================================================

# --- 10.2 Construir nivel_riesgo en la Matriz B ---
def _nivel_riesgo(row):
    if row["fallecidos"] > 0:
        return "fatal"
    if row["lesionados"] > 0:
        return "con_lesionados"
    return "solo_danos"

matriz_b["nivel_riesgo"] = matriz_b.apply(_nivel_riesgo, axis=1)

# Orden lógico de severidad (categórica ordenada, útil para reportes)
orden = ["solo_danos", "con_lesionados", "fatal"]
matriz_b["nivel_riesgo"] = pd.Categorical(
    matriz_b["nivel_riesgo"], categories=orden, ordered=True
)

# --- Verificación e integridad ---
print("TARGET nivel_riesgo (Matriz B — 3 clases)")
print("-" * 55)
vc = matriz_b["nivel_riesgo"].value_counts().reindex(orden)
tabla = pd.DataFrame({"n": vc, "pct": (100 * vc / len(matriz_b)).round(2)})
print(tabla.to_string())
print(f"\n  Total siniestros: {len(matriz_b):,}")

# Chequeo 1: exhaustivo (sin nulos)
assert matriz_b["nivel_riesgo"].isna().sum() == 0, "hay siniestros sin clase asignada"

# Chequeo 2: 'solo_danos' debe coincidir con los que no tienen víctimas
solo_danos_n = int((matriz_b["nivel_riesgo"] == "solo_danos").sum())
sin_vic_n = int(((matriz_b["fallecidos"] == 0) & (matriz_b["lesionados"] == 0)).sum())
print(f"\n  solo_danos = {solo_danos_n:,} | sin víctimas (fall=0 & les=0) = {sin_vic_n:,}")
assert solo_danos_n == sin_vic_n, "desajuste entre solo_danos y siniestros sin víctimas"

# Chequeo 3: coherencia con la severidad agregada de la grilla (Matriz A)
fatal_n = int((matriz_b["nivel_riesgo"] == "fatal").sum())
sin_fallecido = int((matriz_b["fallecidos"] > 0).sum())
print(f"  fatal = {fatal_n:,} | siniestros con fallecidos>0 = {sin_fallecido:,}")
assert fatal_n == sin_fallecido, "desajuste en clase fatal"

# Desbalance (ratio mayor/menor clase)
ratio = vc.max() / vc.min()
print(f"\n  Desbalance (clase mayoritaria / minoritaria): {ratio:.1f}x")
print("  >> OK: nivel_riesgo construido, exhaustivo y consistente.")

TARGET nivel_riesgo (Matriz B — 3 clases)
-------------------------------------------------------
                    n    pct
nivel_riesgo                
solo_danos      46841  52.08
con_lesionados  37458  41.64
fatal            5647   6.28

  Total siniestros: 89,946

  solo_danos = 46,841 | sin víctimas (fall=0 & les=0) = 46,841
  fatal = 5,647 | siniestros con fallecidos>0 = 5,647

  Desbalance (clase mayoritaria / minoritaria): 8.3x
  >> OK: nivel_riesgo construido, exhaustivo y consistente.


## 11. Colinealidad y selección de variables

La colinealidad —predictores que aportan información redundante— infla la varianza de los
coeficientes en modelos lineales y dificulta interpretar el efecto individual de cada
variable. Es especialmente relevante para la **Matriz A**, cuyo modelo (Poisson / Binomial
Negativa) es lineal en los predictores y busca interpretabilidad. Se analiza aquí y se
resuelve la decisión pendiente de la Sección 5.3 (`mes` vs `es_lluviosa`).

### 11.1 Diagnóstico: correlación y VIF sobre las features de la Matriz A

Se evalúan las features numéricas y binarias candidatas de la Matriz A con dos herramientas
complementarias:

- **Matriz de correlación** (Pearson): detecta pares redundantes.
- **VIF (Variance Inflation Factor):** mide cuánto se infla la varianza de cada variable
  por su relación con *todas* las demás. Regla práctica: VIF > 5 indica colinealidad
  preocupante; VIF > 10, severa.

In [41]:
# =============================================================
# Sección 11.1 — Diagnóstico de colinealidad (Matriz A)
# Matriz de correlación + VIF sobre las features numéricas/binarias candidatas.
# =============================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Features numéricas/binarias candidatas de la Matriz A
feats_a = [
    "mes", "es_lluviosa", "es_finde", "dia_semana", "periodo_agostino",
    "es_feriado", "prcp_mensual", "pct_urbano", "poblacion_baja", "log_exposicion",
]

X = matriz_a[feats_a].astype(float)

# --- Matriz de correlación ---
print("MATRIZ DE CORRELACIÓN (features Matriz A)")
print("-" * 55)
corr = X.corr()
print(corr.round(2).to_string())

# Pares con |correlación| alta (excluyendo la diagonal)
print("\nPares con |r| > 0.30:")
for i in range(len(feats_a)):
    for j in range(i + 1, len(feats_a)):
        r = corr.iloc[i, j]
        if abs(r) > 0.30:
            print(f"  {feats_a[i]:<16} ~ {feats_a[j]:<16}: r = {r:+.3f}")

# --- VIF ---
Xc = add_constant(X)
vif = pd.DataFrame({
    "variable": Xc.columns,
    "VIF": [variance_inflation_factor(Xc.values, k) for k in range(Xc.shape[1])],
})
vif = vif[vif["variable"] != "const"].sort_values("VIF", ascending=False)
print("\nVIF (Variance Inflation Factor):")
print(vif.to_string(index=False))

MATRIZ DE CORRELACIÓN (features Matriz A)
-------------------------------------------------------
                   mes  es_lluviosa  es_finde  dia_semana  periodo_agostino  es_feriado  prcp_mensual  pct_urbano  poblacion_baja  log_exposicion
mes               1.00         0.33      0.00        0.00              0.05       -0.01          0.32        0.00            0.00            0.00
es_lluviosa       0.33         1.00     -0.00       -0.00              0.10       -0.01          0.68        0.00           -0.00            0.00
es_finde          0.00        -0.00      1.00        0.47              0.01        0.03         -0.00       -0.00            0.00            0.00
dia_semana        0.00        -0.00      0.47        1.00              0.02        0.04          0.00       -0.00            0.00           -0.00
periodo_agostino  0.05         0.10      0.01        0.02              1.00        0.13          0.04        0.00           -0.00            0.00
es_feriado       -0.01    

### 11.2 Decisión `mes` vs `es_lluviosa` y features finales

Ambas se construyeron en la Sección 5 sabiendo que son redundantes por construcción
(`es_lluviosa` se deriva de `mes`). El diagnóstico confirma la magnitud (r ≈ 0.33) y se
toma la decisión: se **conserva `es_lluviosa`** y se **descarta `mes`** del modelo de la
Matriz A, por ser más parsimoniosa (una binaria vs 12 niveles), directamente interpretable
y alineada con el hallazgo de NB02 (+16.4% en temporada lluviosa). `mes` permanece en el
dataset como columna auxiliar (no se elimina del CSV), pero se marca como **no-feature**.

**Otros pares revisados (se conservan todos).** El diagnóstico mostró VIF de todas las
variables por debajo de 2.2 —muy lejos del umbral de 5—, por lo que **no se elimina nada
más por colinealidad**:

- `es_lluviosa ~ prcp_mensual` (r ≈ 0.68): se **conservan ambas**. Capturan matices
  distintos —temporada (binaria) vs intensidad de lluvia en mm (continua)— y el VIF < 2 lo
  tolera sin problema.
- `pct_urbano`, `poblacion_baja`, `log_exposicion` (correlaciones ~0.4–0.6 entre sí): se
  **conservan las tres**. Están relacionadas por el tamaño/urbanización del distrito, pero
  con VIF < 2.2. Además `log_exposicion` entra como **offset** (coeficiente fijo en 1), no
  como feature, por lo que su colinealidad no afecta la estimación de los coeficientes.

**Separación offset vs features (handoff a NB04).** `log_exposicion` se aparta
explícitamente de la lista de predictoras: va como offset del modelo de conteo. Las 8
features restantes conforman `FEATURES_MODELO_A`.

In [42]:
# =============================================================
# Sección 11.2 — Decisión mes vs es_lluviosa y definición de features finales
# Se descarta 'mes' (redundante). log_exposicion se separa como OFFSET.
# =============================================================
r_mes_lluv  = corr.loc["mes", "es_lluviosa"]
r_lluv_prcp = corr.loc["es_lluviosa", "prcp_mensual"]

print("DECISIÓN DE SELECCIÓN")
print("-" * 55)
print(f"  r(mes, es_lluviosa)          = {r_mes_lluv:+.3f}  -> se descarta 'mes'")
print(f"  r(es_lluviosa, prcp_mensual) = {r_lluv_prcp:+.3f}  -> se conservan ambas (VIF<2)")
print(f"  VIF máximo observado         = {vif['VIF'].max():.2f}  (< 5: sin colinealidad severa)")

# Offset del modelo A (coef fijo=1), separado de las predictoras
OFFSET_A = "log_exposicion"

# Features predictoras del modelo A (sin 'mes' y sin el offset)
FEATURES_MODELO_A = [
    "es_lluviosa", "es_finde", "dia_semana", "periodo_agostino",
    "es_feriado", "prcp_mensual", "pct_urbano", "poblacion_baja",
]

print(f"\n  Offset del modelo A  : {OFFSET_A}")
print(f"  Features del modelo A ({len(FEATURES_MODELO_A)}):")
for feat in FEATURES_MODELO_A:
    print(f"    - {feat}")
print(f"\n  'mes' queda en el CSV como columna auxiliar (no-feature).")

DECISIÓN DE SELECCIÓN
-------------------------------------------------------
  r(mes, es_lluviosa)          = +0.327  -> se descarta 'mes'
  r(es_lluviosa, prcp_mensual) = +0.676  -> se conservan ambas (VIF<2)
  VIF máximo observado         = 2.17  (< 5: sin colinealidad severa)

  Offset del modelo A  : log_exposicion
  Features del modelo A (8):
    - es_lluviosa
    - es_finde
    - dia_semana
    - periodo_agostino
    - es_feriado
    - prcp_mensual
    - pct_urbano
    - poblacion_baja

  'mes' queda en el CSV como columna auxiliar (no-feature).


### 11.3 Nota sobre la Matriz B

En la Matriz B el clasificador previsto (Random Forest) es **robusto a la colinealidad** y
sus predictores son mayormente categóricos, por lo que **no se aplica un VIF exhaustivo**.
La colinealidad no compromete el desempeño de un modelo de árboles; a lo sumo reparte la
importancia entre variables correlacionadas, lo que se tendrá en cuenta al interpretar el
análisis SHAP/LIME en NB04.

## 12. Ensamblaje y split temporal

Última etapa de preprocesamiento: se seleccionan las columnas finales de cada matriz, se
particiona en train/test respetando el orden temporal (sin fuga de futuro) y se guardan los
CSV en `data/processed/` para consumo de NB04.

### 12.1 Selección de columnas finales

Cada matriz conserva: sus **identificadores** (para trazabilidad y joins futuros), sus
**features**, su **target**, y las **columnas auxiliares** relevantes (marcadas como
no-feature pero útiles para análisis). No se eliminan columnas informativas; solo se ordenan
y se descartan las auxiliares internas del proceso (las que empiezan con `_`).

**Matriz A (frecuencia):**
- Identificadores: `distrito_key`, `departamento_key`, `fecha`, `franja_horaria`, `anio_mes`, `estacion_asignada`
- Target: `n_siniestros`
- Features del modelo: las 8 de `FEATURES_MODELO_A` + offset `log_exposicion`
- Auxiliares (no-feature): `mes`, `poblacion`, `fallecidos`, `lesionados`, `prcp_imputada`

**Matriz B (severidad):**
- Identificadores: `id_siniestro`, `distrito_key`, `departamento_key`, `fecha`, `franja_horaria`
- Target: `nivel_riesgo`
- Features predictoras: categóricas del siniestro (`factor_causante_grp`, `tipo_siniestro`,
  `tipo_vehiculo`, `condicion_via`) + ex-ante reutilizables (calendario/clima/exposición)
- Descriptivas (no-feature del clasificador de 3 clases): `grupo_vulnerable`, `rango_etario`,
  `fallecidos`, `lesionados`

In [43]:
# =============================================================
# Sección 12.1 — Selección de columnas finales de cada matriz
# Matriz A: ya tiene todas sus features. Solo se ordenan y se quitan auxiliares '_'.
# Matriz B: se le agregan las features de calendario (recalculadas desde su fecha),
#           luego se seleccionan columnas.
# =============================================================

# --- Matriz B: agregar features de calendario desde su propia fecha ---
fb = matriz_b["fecha"]
matriz_b["mes"]              = fb.dt.month
matriz_b["es_lluviosa"]      = fb.dt.month.between(5, 10).astype(int)
matriz_b["dia_semana"]       = fb.dt.dayofweek
matriz_b["es_finde"]         = fb.dt.dayofweek.isin([4, 5]).astype(int)
matriz_b["periodo_agostino"] = ((fb.dt.month == 8) & (fb.dt.day.between(3, 6))).astype(int)
# es_feriado en B: nacional a todos + local SS solo a san salvador (misma lógica que A)
comp_nac_b   = matriz_b["fecha"].dt.normalize().isin(fechas_nacionales)
comp_loc_b   = (matriz_b["fecha"].dt.normalize().isin(fechas_local_ss)
                & (matriz_b["distrito_key"] == "san salvador"))
matriz_b["es_feriado"] = (comp_nac_b | comp_loc_b).astype(int)

# --- Columnas finales Matriz A ---
cols_a = [
    # identificadores
    "distrito_key", "departamento_key", "fecha", "franja_horaria",
    "anio_mes", "estacion_asignada",
    # target
    "n_siniestros",
    # features del modelo (8)
    *FEATURES_MODELO_A,
    # offset
    OFFSET_A,
    # auxiliares (no-feature, útiles para análisis/trazabilidad)
    "mes", "poblacion", "fallecidos", "lesionados", "prcp_imputada",
]
matriz_a_final = matriz_a[cols_a].copy()

# --- Columnas finales Matriz B ---
cols_b = [
    # identificadores
    "id_siniestro", "distrito_key", "departamento_key", "fecha", "franja_horaria",
    # target
    "nivel_riesgo",
    # features predictoras: categóricas del siniestro
    "factor_causante_grp", "tipo_siniestro", "tipo_vehiculo", "condicion_via",
    # features predictoras: calendario (ex-ante)
    "es_lluviosa", "es_finde", "dia_semana", "periodo_agostino", "es_feriado",
    # descriptivas (no-feature del clasificador de 3 clases)
    "grupo_vulnerable", "rango_etario", "fallecidos", "lesionados",
    # auxiliar
    "mes",
]
matriz_b_final = matriz_b[cols_b].copy()

# --- Verificación ---
print("SELECCIÓN DE COLUMNAS FINALES")
print("-" * 55)
print(f"  Matriz A : {matriz_a_final.shape[0]:,} filas × {matriz_a_final.shape[1]} columnas")
print(f"  Matriz B : {matriz_b_final.shape[0]:,} filas × {matriz_b_final.shape[1]} columnas")

# Chequeo: sin columnas auxiliares internas ('_...') ni nulos en columnas clave
assert not any(c.startswith("_") for c in matriz_a_final.columns), "quedó columna '_' en A"
assert not any(c.startswith("_") for c in matriz_b_final.columns), "quedó columna '_' en B"
assert matriz_a_final["n_siniestros"].isna().sum() == 0, "nulos en target A"
assert matriz_b_final["nivel_riesgo"].isna().sum() == 0, "nulos en target B"

print("\n  Columnas Matriz A:")
print("   ", list(matriz_a_final.columns))
print("\n  Columnas Matriz B:")
print("   ", list(matriz_b_final.columns))

SELECCIÓN DE COLUMNAS FINALES
-------------------------------------------------------
  Matriz A : 676,504 filas × 21 columnas
  Matriz B : 89,946 filas × 20 columnas

  Columnas Matriz A:
    ['distrito_key', 'departamento_key', 'fecha', 'franja_horaria', 'anio_mes', 'estacion_asignada', 'n_siniestros', 'es_lluviosa', 'es_finde', 'dia_semana', 'periodo_agostino', 'es_feriado', 'prcp_mensual', 'pct_urbano', 'poblacion_baja', 'log_exposicion', 'mes', 'poblacion', 'fallecidos', 'lesionados', 'prcp_imputada']

  Columnas Matriz B:
    ['id_siniestro', 'distrito_key', 'departamento_key', 'fecha', 'franja_horaria', 'nivel_riesgo', 'factor_causante_grp', 'tipo_siniestro', 'tipo_vehiculo', 'condicion_via', 'es_lluviosa', 'es_finde', 'dia_semana', 'periodo_agostino', 'es_feriado', 'grupo_vulnerable', 'rango_etario', 'fallecidos', 'lesionados', 'mes']


### 12.2 Split temporal (train / test)

La partición respeta el **orden temporal** para evitar fuga de información de períodos
futuros hacia el entrenamiento —requisito explícito del anteproyecto—. No se usa un split
aleatorio: en un problema con estructura temporal, mezclar fechas permitiría al modelo
"ver el futuro" durante el entrenamiento, inflando artificialmente su desempeño.

**Corte elegido:**

| Partición | Período | Rol |
|---|---|---|
| **Train** | 2022 – 2024 (3 años completos) | el modelo aprende de estos años |
| **Test**  | 2025 – 2026 (2025 completo + ene–jun 2026) | evaluación en período futuro no visto |

**Por qué este corte y no otro.** El conjunto de prueba **debe** contener el fenómeno más
relevante del proyecto: el pico agostino. Con test = 2025–2026, el test incluye **agosto
2025 completo** y todo el segundo semestre de 2025, evaluando el efecto de
`periodo_agostino`, los feriados del segundo semestre y ambas temporadas climáticas. Un
corte alternativo con test solo en 2026 (parcial ene–jun) habría dejado el test **ciego al
pico agostino** y al segundo semestre. El costo —que train tenga 3 agostos en vez de 4— es
irrelevante para aprender el patrón.

El mismo corte se aplica a **ambas matrices** (A y B), de modo que los registros de prueba
correspondan al mismo período en las dos.

**Regla anti-fuga.** El split se hace por fecha: ninguna celda/siniestro de 2025–2026 entra
al train, y ninguna variable se calcula usando información del período de test.

In [44]:
# =============================================================
# Sección 12.2 — Split temporal train/test (mismo corte para A y B)
# Train: 2022-2024  |  Test: 2025-2026. Corte por año de la fecha.
# =============================================================
ANIO_CORTE = 2025  # test = fechas con año >= 2025; train = año < 2025

def split_temporal(df, etiqueta):
    train = df[df["fecha"].dt.year < ANIO_CORTE].copy()
    test  = df[df["fecha"].dt.year >= ANIO_CORTE].copy()
    print(f"\n[{etiqueta}]")
    print(f"  Train (2022-2024): {len(train):>9,} filas  "
          f"({100*len(train)/len(df):.1f}%)  "
          f"[{train['fecha'].min().date()} … {train['fecha'].max().date()}]")
    print(f"  Test  (2025-2026): {len(test):>9,} filas  "
          f"({100*len(test)/len(df):.1f}%)  "
          f"[{test['fecha'].min().date()} … {test['fecha'].max().date()}]")
    # Anti-fuga: sin solape temporal
    assert train["fecha"].max() < test["fecha"].min(), "solape temporal train/test"
    return train, test

print("SPLIT TEMPORAL (train 2022-2024 / test 2025-2026)")
print("-" * 55)

A_train, A_test = split_temporal(matriz_a_final, "Matriz A — frecuencia")
B_train, B_test = split_temporal(matriz_b_final, "Matriz B — severidad")

# --- Verificación de que el pico agostino está presente en el test de A ---
ago_test = A_test[(A_test["fecha"].dt.month == 8) & (A_test["periodo_agostino"] == 1)]
print(f"\n  Celdas con periodo_agostino en TEST de A: {len(ago_test):,} "
      f"(confirma que el test evalúa el pico agostino)")
assert len(ago_test) > 0, "el test no contiene el período agostino (revisar corte)"

SPLIT TEMPORAL (train 2022-2024 / test 2025-2026)
-------------------------------------------------------

[Matriz A — frecuencia]
  Train (2022-2024):   451,552 filas  (66.7%)  [2022-01-01 … 2024-12-31]
  Test  (2025-2026):   224,952 filas  (33.3%)  [2025-01-01 … 2026-06-30]

[Matriz B — severidad]
  Train (2022-2024):    56,172 filas  (62.5%)  [2022-01-01 … 2024-12-31]
  Test  (2025-2026):    33,774 filas  (37.5%)  [2025-01-01 … 2026-06-30]

  Celdas con periodo_agostino en TEST de A: 1,648 (confirma que el test evalúa el pico agostino)


### 12.3 Guardado de las matrices en `data/processed/`

Se persisten las dos matrices como CSV en `data/processed/`, listas para NB04. En lugar de
generar cuatro archivos separados (train/test por matriz), cada matriz se guarda **completa
con una columna `split`** que marca cada fila como `train` o `test`. Ventajas: menos
archivos que mantener sincronizados, inspección más simple, y el corte temporal queda
documentado dentro del propio dato (NB04 filtra con `df[df.split == "train"]`).

**Archivos generados:**
- `matriz_frecuencia.csv` — Matriz A (grilla distrito×fecha×franja, target `n_siniestros`).
- `matriz_severidad.csv` — Matriz B (siniestro individual, target `nivel_riesgo`).

**Nota sobre CSV y tipos.** El formato CSV no preserva tipos (fechas y categóricas se
releen como texto). Para el handoff a NB04 se documentan los tipos clave en la Sección 13;
al recargar, NB04 debe parsear `fecha` como datetime y tratar las categóricas como tales.
Se versionan en el repositorio (según `.gitignore`) como evidencia reproducible.

In [45]:
# =============================================================
# Sección 12.3 — Guardado de las matrices en data/processed/
# Cada matriz se guarda completa con columna 'split' (train/test).
# =============================================================

# --- Añadir columna 'split' a cada matriz completa ---
matriz_a_final["split"] = (
    matriz_a_final["fecha"].dt.year.ge(ANIO_CORTE).map({False: "train", True: "test"})
)
matriz_b_final["split"] = (
    matriz_b_final["fecha"].dt.year.ge(ANIO_CORTE).map({False: "train", True: "test"})
)

# --- Rutas de salida ---
ruta_a = cfg.PROCESSED_DIR / "matriz_frecuencia.csv"
ruta_b = cfg.PROCESSED_DIR / "matriz_severidad.csv"

# --- Guardar ---
matriz_a_final.to_csv(ruta_a, index=False, encoding="utf-8")
matriz_b_final.to_csv(ruta_b, index=False, encoding="utf-8")

print("GUARDADO EN data/processed/")
print("-" * 55)
for ruta, df, nombre in [(ruta_a, matriz_a_final, "matriz_frecuencia"),
                         (ruta_b, matriz_b_final, "matriz_severidad")]:
    existe = ruta.exists()
    mb = ruta.stat().st_size / 1e6 if existe else 0
    print(f"  {nombre:<20}: {'OK' if existe else 'FALTA'}  "
          f"{df.shape[0]:>7,} filas × {df.shape[1]} cols  ({mb:.1f} MB)")
    print(f"    split -> {df['split'].value_counts().to_dict()}")

# --- Verificación de reproducibilidad: releer y confirmar dimensiones ---
print("\nVerificación de re-lectura:")
a_check = pd.read_csv(ruta_a)
b_check = pd.read_csv(ruta_b)
assert a_check.shape == matriz_a_final.shape, "matriz_frecuencia relee con dimensión distinta"
assert b_check.shape == matriz_b_final.shape, "matriz_severidad relee con dimensión distinta"
print(f"  matriz_frecuencia: relee {a_check.shape[0]:,} × {a_check.shape[1]} — OK")
print(f"  matriz_severidad : relee {b_check.shape[0]:,} × {b_check.shape[1]} — OK")
print("\n>> Matrices guardadas y verificadas. Listas para NB04.")

GUARDADO EN data/processed/
-------------------------------------------------------
  matriz_frecuencia   : OK  676,504 filas × 22 cols  (84.3 MB)
    split -> {'train': 451552, 'test': 224952}
  matriz_severidad    : OK   89,946 filas × 21 cols  (15.3 MB)
    split -> {'train': 56172, 'test': 33774}

Verificación de re-lectura:
  matriz_frecuencia: relee 676,504 × 22 — OK
  matriz_severidad : relee 89,946 × 21 — OK

>> Matrices guardadas y verificadas. Listas para NB04.


## 13. Resumen y handoff a NB04

NB03 transformó los datos crudos validados en NB01 y caracterizados en NB02 en dos matrices
de features listas para modelar, guardadas en `data/processed/`.

### 13.1 Productos generados

| Matriz | Archivo | Unidad | Filas | Target | Modelo (NB04) |
|---|---|---|---|---|---|
| **A — Frecuencia** | `matriz_frecuencia.csv` | distrito×fecha×franja | 676 504 | `n_siniestros` | Poisson / Binomial Negativa |
| **B — Severidad** | `matriz_severidad.csv` | siniestro individual | 89 946 | `nivel_riesgo` (3 clases) | Clasificación (Random Forest) |

Ambas incluyen la columna `split` (train = 2022–2024, test = 2025–2026).

In [46]:
# =============================================================
# Sección 13.1 — Verificación de los productos generados en disco
# =============================================================
print("PRODUCTOS DE NB03 EN data/processed/")
print("-" * 55)
for nombre, ruta, target in [
    ("Matriz A (frecuencia)", cfg.PROCESSED_DIR / "matriz_frecuencia.csv", "n_siniestros"),
    ("Matriz B (severidad)",  cfg.PROCESSED_DIR / "matriz_severidad.csv",  "nivel_riesgo"),
]:
    df = pd.read_csv(ruta, parse_dates=["fecha"])
    print(f"\n{nombre}")
    print(f"  Archivo : {ruta.name}  ({ruta.stat().st_size/1e6:.1f} MB)")
    print(f"  Forma   : {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"  Target  : {target}")
    print(f"  Split   : {df['split'].value_counts().to_dict()}")
    print(f"  Fecha   : {df['fecha'].min().date()} … {df['fecha'].max().date()} "
          f"(dtype={df['fecha'].dtype})")

PRODUCTOS DE NB03 EN data/processed/
-------------------------------------------------------

Matriz A (frecuencia)
  Archivo : matriz_frecuencia.csv  (84.3 MB)
  Forma   : 676,504 filas × 22 columnas
  Target  : n_siniestros
  Split   : {'train': 451552, 'test': 224952}
  Fecha   : 2022-01-01 … 2026-06-30 (dtype=datetime64[ns])

Matriz B (severidad)
  Archivo : matriz_severidad.csv  (15.3 MB)
  Forma   : 89,946 filas × 21 columnas
  Target  : nivel_riesgo
  Split   : {'train': 56172, 'test': 33774}
  Fecha   : 2022-01-01 … 2026-06-30 (dtype=datetime64[ns])


### 13.2 Cómo cargar las matrices en NB04 (importante)

El formato CSV no preserva tipos. Al cargar en NB04, **parsear la fecha explícitamente**:

```python
matriz_a = pd.read_csv(cfg.PROCESSED_DIR / "matriz_frecuencia.csv", parse_dates=["fecha"])
matriz_b = pd.read_csv(cfg.PROCESSED_DIR / "matriz_severidad.csv", parse_dates=["fecha"])
```

Las columnas categóricas (`franja_horaria`, `factor_causante_grp`, `tipo_siniestro`,
`tipo_vehiculo`, `condicion_via`, `grupo_vulnerable`, `rango_etario`, `nivel_riesgo`) se
releen como texto; convertir a `category` o codificar según lo requiera cada modelo.

### 13.3 Features del modelo de frecuencia (Matriz A)

- **Target:** `n_siniestros`
- **Offset:** `log_exposicion` (coeficiente fijo = 1; el modelo predice *tasa*, no conteo)
- **Features (8):** `es_lluviosa`, `es_finde`, `dia_semana`, `periodo_agostino`,
  `es_feriado`, `prcp_mensual`, `pct_urbano`, `poblacion_baja`
- **No-features** (auxiliares, en el CSV pero fuera del modelo): `mes`, `poblacion`,
  `fallecidos`, `lesionados`, `prcp_imputada`, identificadores

In [47]:
# =============================================================
# Sección 13.3 — Verificar que las features del modelo A existen y son válidas
# =============================================================
a = pd.read_csv(cfg.PROCESSED_DIR / "matriz_frecuencia.csv", parse_dates=["fecha"])

FEATURES_MODELO_A = [
    "es_lluviosa", "es_finde", "dia_semana", "periodo_agostino",
    "es_feriado", "prcp_mensual", "pct_urbano", "poblacion_baja",
]
OFFSET_A = "log_exposicion"

# Chequeos: existen, sin nulos, y el offset es positivo (log de población > 0)
faltan = [c for c in FEATURES_MODELO_A + [OFFSET_A, "n_siniestros"] if c not in a.columns]
assert not faltan, f"faltan columnas en matriz A: {faltan}"
assert a[FEATURES_MODELO_A + [OFFSET_A]].isna().sum().sum() == 0, "hay nulos en features de A"

print("MATRIZ A — features del modelo verificadas")
print("-" * 55)
print(f"  Target  : n_siniestros  (media={a['n_siniestros'].mean():.4f})")
print(f"  Offset  : {OFFSET_A}  (rango {a[OFFSET_A].min():.2f}–{a[OFFSET_A].max():.2f})")
print(f"  Features ({len(FEATURES_MODELO_A)}): {FEATURES_MODELO_A}")
print("  >> Sin nulos, listas para NB04.")

MATRIZ A — features del modelo verificadas
-------------------------------------------------------
  Target  : n_siniestros  (media=0.1330)
  Offset  : log_exposicion  (rango 7.65–12.71)
  Features (8): ['es_lluviosa', 'es_finde', 'dia_semana', 'periodo_agostino', 'es_feriado', 'prcp_mensual', 'pct_urbano', 'poblacion_baja']
  >> Sin nulos, listas para NB04.


### 13.4 Features del modelo de severidad (Matriz B)

- **Target:** `nivel_riesgo` (solo_danos / con_lesionados / fatal; desbalance ≈ 8.3×)
- **Features predictoras:** `factor_causante_grp`, `tipo_siniestro`, `tipo_vehiculo`,
  `condicion_via` + calendario (`es_lluviosa`, `es_finde`, `dia_semana`,
  `periodo_agostino`, `es_feriado`)
- **No-features del clasificador de 3 clases** (descriptivas, por regla anti-leakage):
  `grupo_vulnerable`, `rango_etario`, `fallecidos`, `lesionados`

In [48]:
# =============================================================
# Sección 13.4 — Verificar features del modelo B y el desbalance del target
# =============================================================
b = pd.read_csv(cfg.PROCESSED_DIR / "matriz_severidad.csv", parse_dates=["fecha"])

FEATURES_MODELO_B = [
    "factor_causante_grp", "tipo_siniestro", "tipo_vehiculo", "condicion_via",
    "es_lluviosa", "es_finde", "dia_semana", "periodo_agostino", "es_feriado",
]
NO_FEATURES_B = ["grupo_vulnerable", "rango_etario", "fallecidos", "lesionados"]

faltan = [c for c in FEATURES_MODELO_B + ["nivel_riesgo"] if c not in b.columns]
assert not faltan, f"faltan columnas en matriz B: {faltan}"

print("MATRIZ B — features del modelo verificadas")
print("-" * 55)
print(f"  Target: nivel_riesgo")
vc = b["nivel_riesgo"].value_counts()
for clase, n in vc.items():
    print(f"    {clase:<16}: {n:>6,} ({100*n/len(b):.2f}%)")
print(f"  Desbalance (mayor/menor): {vc.max()/vc.min():.1f}x")
print(f"\n  Features predictoras ({len(FEATURES_MODELO_B)}): {FEATURES_MODELO_B}")
print(f"  No-features (anti-leakage): {NO_FEATURES_B}")
print("  >> Listas para NB04.")

MATRIZ B — features del modelo verificadas
-------------------------------------------------------
  Target: nivel_riesgo
    solo_danos      : 46,841 (52.08%)
    con_lesionados  : 37,458 (41.64%)
    fatal           :  5,647 (6.28%)
  Desbalance (mayor/menor): 8.3x

  Features predictoras (9): ['factor_causante_grp', 'tipo_siniestro', 'tipo_vehiculo', 'condicion_via', 'es_lluviosa', 'es_finde', 'dia_semana', 'periodo_agostino', 'es_feriado']
  No-features (anti-leakage): ['grupo_vulnerable', 'rango_etario', 'fallecidos', 'lesionados']
  >> Listas para NB04.


### 13.5 Recordatorios

- **Sobredispersión + 89.7% ceros** en la Matriz A → comparar Poisson vs Binomial Negativa
  por AIC; considerar mención de modelos zero-inflated (ZIP/ZINB) como refinamiento.
- **Desbalance de clases** en la Matriz B → usar F1 (macro/por clase), no solo accuracy;
  considerar class weights o remuestreo.
- **Distritos de población baja** (`poblacion_baja = 1`, 12 distritos): decidir si filtrar
  o conservar en el modelado (el flag ya está; la decisión es de NB04).
- **Interpretabilidad:** SHAP/LIME sobre los modelos, como pide la rúbrica (10 pts).